# Stage 10 — Release Package Assembly vFINAL

**ZERO-SCIENCE RELEASE ENGINEERING ONLY**

This notebook assembles a release from **already executed Save-Version outputs** of the locked canonical chain:

`01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09`

It MUST NOT rerun, retrain, recalibrate, bootstrap, recompute inferential statistics, regenerate figures, or modify any canonical scientific notebook.

### Fail-closed rules
- Canonical notebook identity is hash-gated.
- Existing stage manifests/checksum contracts are verified before copying.
- Any required artifact absence or hash mismatch is a hard stop.
- Stages 05–08 use downstream-pinned canonical hashes.
- Stage 09 must be a unique, payload-sealed, PASS sensitivity output with the exact scientific-role label.
- Stage 03 transformer checkpoints are explicitly separated from the core release.
- Unverified dataset/model redistribution licenses are never invented.
- Historical/intermediate notebooks are archived and labeled non-canonical.
- Historical environment records are kept distinct from the Stage 10 assembly environment.

**This notebook intentionally contains no scientific computation.**


In [1]:

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import csv
import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import re
import shutil
import sys
import zipfile

STAGE_ID = "10"
STAGE_NAME = "10_RELEASE_PACKAGE_ASSEMBLY_vFINAL"
PROTOCOL_VERSION = "SWARABYANJAN-CROSSPLATFORM-v1.0.0"
RELEASE_SCHEMA_VERSION = 1

CANONICAL_NOTEBOOKS = [
    "01-dataset-acquisition-and-freeze.ipynb",
    "02-p0-integrity-provenance-leakage-audit.ipynb",
    "03-primary-model-development.ipynb",
    "04-calibration.ipynb",
    "05-external-validation-and-h3-vfinal.ipynb",
    "06-fixed-budget-reliability-evaluation-vfinal.ipynb",
    "07-h3-h4-bridge-exploratory-vfinal.ipynb",
    "08-manuscript-figures-and-tables-vfinal.ipynb",
    "09-source-publisher-cluster-sensitivity-vfinal.ipynb"
]
CANONICAL_NOTEBOOK_SHA256 = {
    "01-dataset-acquisition-and-freeze.ipynb": "627e38c7fd99c53b483bc109404b03e781b060c58215e639121838cb63abe8cf",
    "02-p0-integrity-provenance-leakage-audit.ipynb": "425357156376ea9b5724fae120ca969f8f2c93c94d0f92cba252461b2835a85f",
    "03-primary-model-development.ipynb": "db17c0ecbc55eb15a9c5e7518616df6494400db093c82a66576cd4911b221f1c",
    "04-calibration.ipynb": "9c80cfe9a16cba71813f7cb539368e674694d820228b66b8392ad83341cd81ea",
    "05-external-validation-and-h3-vfinal.ipynb": "ee258026c5d6634c6b0c0bb66553971d06a9148e36ff13a4ea7f4886905a81cf",
    "06-fixed-budget-reliability-evaluation-vfinal.ipynb": "4fa809490ffd9a08791fe2da1fab5b1056b51c68443bd97060577d05b1012c43",
    "07-h3-h4-bridge-exploratory-vfinal.ipynb": "e3c961eff6a380eb86f919dc598b5298cadb6d749548e3f33fb97c5b7c8f4967",
    "08-manuscript-figures-and-tables-vfinal.ipynb": "8766604fe554acb362811d35314982d62f413a7489068aeca8b619915b479c62",
    "09-source-publisher-cluster-sensitivity-vfinal.ipynb": "d4b267d17debe23f470343dcb0d949c274a31510d3acc5178ee6a18f2adcb64c"
}
HISTORICAL_NOTEBOOKS = [
    "05-external-validation.ipynb",
    "05-h3-statistical-repair-only.ipynb",
    "06-fixed-budget-reliability-evaluation.ipynb"
]
HISTORICAL_NOTEBOOK_SHA256 = {
    "05-external-validation.ipynb": "685b2be79ec58696f0d9f0e022dd7b7ef32a56c2ba2807a463d598ce3d804bdc",
    "05-h3-statistical-repair-only.ipynb": "5891a31b23564e1b502638422d837149c915c68fbd4345f10a86fe8ce0e8f177",
    "06-fixed-budget-reliability-evaluation.ipynb": "31a18d08ff612cb49f37822bd1705445743a8134fa58e8e957f5d7f21038df69"
}

CANONICAL_STAGE_DIRS = {
    "01": "01_frozen_banglabait",
    "02": "02_cleaning",
    "03": "03_modeling",
    "04": "04_calibration",
    "05": "05_external_validation_and_H3",
    "06": "06_fixed_budget_reliability",
    "07": "07_h3_h4_bridge_exploratory",
    "08": "08_manuscript_figures_and_tables",
    "09": "09_source_site_cluster_sensitivity",
}

CANONICAL_NOTEBOOK_RELEASE_NAMES = {
    "01-dataset-acquisition-and-freeze.ipynb": "01-dataset-acquisition-and-freeze.ipynb",
    "02-p0-integrity-provenance-leakage-audit.ipynb": "02-p0-integrity-provenance-leakage-audit.ipynb",
    "03-primary-model-development.ipynb": "03-primary-model-development.ipynb",
    "04-calibration.ipynb": "04-calibration.ipynb",
    "05-external-validation-and-h3-vfinal.ipynb": "05_EXTERNAL_VALIDATION_H3_vFINAL.ipynb",
    "06-fixed-budget-reliability-evaluation-vfinal.ipynb": "06_FIXED_BUDGET_RELIABILITY_EVALUATION_vFINAL.ipynb",
    "07-h3-h4-bridge-exploratory-vfinal.ipynb": "07_H3_H4_BRIDGE_EXPLORATORY_vFINAL.ipynb",
    "08-manuscript-figures-and-tables-vfinal.ipynb": "08_MANUSCRIPT_FIGURES_AND_TABLES_vFINAL.ipynb",
    "09-source-publisher-cluster-sensitivity-vfinal.ipynb": "09_SOURCE_SITE_CLUSTER_SENSITIVITY_vFINAL.ipynb",
}

EXPECTED_FROZEN_ZIP_SHA256 = "e0de5c224f406fe22d8ad30804217465f789c49c70e1d3afbe00afd28deb7ccf"
EXPECTED_BANGLABAIT_TEST_SHA256 = "68176be61ba8dd01f9f78b6aea2b84c198236c89de5abc98db8c49c496ebd996"
EXPECTED_STAGE02_METHODOLOGY_SHA256 = "fcf658785c8a91ffa31a08b7861dbed07aac26f06b52d133ce257f3610d3ce6b"

EXPECTED_STAGE05 = {
    "manifest_name": "STAGE05_ARTIFACT_MANIFEST.json",
    "manifest_sha256": "7246bc3a49a5f8d8472005cea9f6062d118431cd67d5dd2c4fd8f2ee31f52c27",
    "checksum_name": "STAGE05_SHA256SUMS.txt",
    "checksum_sha256": "7171a0e581d8fb4bde2a630f4de631d74962954ee3890b8123d78a720823450a",
    "status_name": "STAGE_STATUS.json",
    "status_sha256": "005399d829bb66870810faa3f140757a7ee3d82eac4bae394fbbe79e701db8e6",
    "canonical_version": "SWARABYANJAN-STAGE05-CANONICAL-vFINAL-1.0.1",
    "prediction_sha256": {
        "LOCKED_SOURCE_TEST_PREDICTIONS.parquet": "df863d7a24f7d451451822890de58613c3295eb0bcc1aa882150137b0aa9880a",
        "BANGLACLICK_PLATFORM_PREDICTIONS.parquet": "5685fd4142a0e8e7f85cd6040cf48042da2368d5f9596a7740c4f9f09c9519d5",
        "BAITBUSTER_HUMAN_PREDICTIONS.parquet": "a3e95820c62de00984a189e2c250f7fffd223ad4b3954db04a5368c7a544b7b9",
    },
}
EXPECTED_STAGE06 = {
    "manifest_name": "REPRODUCIBILITY_MANIFEST.json",
    "manifest_sha256": "5a2962014445c723379b07fcf02e601644818c36e6c15fb7ee9478ca34711285",
    "checksum_name": "STAGE06_SHA256SUMS.txt",
    "checksum_sha256": "5f8d0f7fc9cde5e0f159bf4c5f4ca8387441675407b4c43ab87a8063afac3beb",
    "status_name": "STAGE_STATUS.json",
    "status_sha256": "769f33b4771f3ec55700384ae6db7e9e2feb115d23717b27cbd9c17e4983538d",
    "h4_sha256": "299fc2e1394c80e1a090ac14d632e0f502b5b779c459e47dcb4e96130b621462",
}
EXPECTED_STAGE07 = {
    "manifest_name": "REPRODUCIBILITY_MANIFEST.json",
    "manifest_sha256": "fa8651ff00a3306dfd48ed55d21df3ab2eef2fbd365db33657f5c2413822e714",
    "checksum_name": "STAGE07_SHA256SUMS.txt",
    "checksum_sha256": "86c8aeb1c4c0062303b44d397fece5908c315ff8860fca59afc624bfdde39f8d",
    "status_name": "STAGE_STATUS.json",
    "status_sha256": "566f063f1bef265c6d377565ee6a22ccc1cbbbaa083f7bad7a89423ec6222280",
}
EXPECTED_STAGE08 = {
    "manifest_name": "MANIFEST.json",
    "manifest_sha256": "b0ac3b392b2e3d1b5136fda5dbdbcf498e1b89c44cc1d920ff597d6c2967523f",
    "checksum_name": "STAGE08_SHA256SUMS.txt",
    "checksum_sha256": "ccd850d3dbe144651bb52e8277308e6438e3e3d869b2cfc232b39db96fed3af6",
    "status_name": "STAGE08_STATUS.json",
    "status_sha256": "d64ee2649a348bad4fd36ee3c4df79fd071b60807b248f0a7a575f5795bf9f33",
}
EXPECTED_STAGE09_SOURCE_ROLE = "sensitivity only; never replaces canonical Stages05/06/07/08"
RELEASE_STAGE09_ROLE = "sensitivity only; never replaces canonical Stages 05/06/07/08"
EXPECTED_STAGE09_SENSITIVITY_VERSION = "SWARABYANJAN-SOURCE-SITE-CLUSTER-SENSITIVITY-v1.0.0"

RAW_DATA_REDISTRIBUTION_VERIFIED = False
COPY_OPTIONAL_LARGE_TRAINING_ARTIFACTS = False

def utc_now():
    return datetime.now(timezone.utc).isoformat()

print("Stage 10 configuration loaded — ZERO-SCIENCE assembly only.")


Stage 10 configuration loaded — ZERO-SCIENCE assembly only.


## 1. Release-engineering utilities

Only filesystem discovery, JSON/CSV parsing, cryptographic hashing, copying, redaction, and documentation are permitted below. No scientific library is imported.


In [2]:

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                return h.hexdigest()
            h.update(block)

def canonical_json_bytes(obj):
    return json.dumps(
        obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
    ).encode("utf-8")

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )

def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(str(text), encoding="utf-8")

def parse_sha256sums(path):
    out = {}
    for raw in Path(path).read_text(encoding="utf-8").splitlines():
        if not raw.strip():
            continue
        parts = raw.split(None, 1)
        if len(parts) != 2 or not re.fullmatch(r"[0-9a-fA-F]{64}", parts[0]):
            raise RuntimeError(f"Malformed checksum line in {path}: {raw!r}")
        rel = parts[1].lstrip("*").strip().replace("\\", "/")
        if rel in out:
            raise RuntimeError(f"Duplicate checksum path in {path}: {rel}")
        out[rel] = parts[0].lower()
    if not out:
        raise RuntimeError(f"Empty checksum contract: {path}")
    return out

def verify_checksum_contract(root, checksum_name, require_all=True):
    root = Path(root)
    sums = parse_sha256sums(root / checksum_name)
    failures = []
    for rel, expected in sums.items():
        p = root / rel
        if not p.is_file():
            failures.append({"path": rel, "reason": "MISSING"})
        else:
            observed = sha256_file(p)
            if observed != expected:
                failures.append({"path": rel, "reason": "HASH_MISMATCH",
                                 "expected": expected, "observed": observed})
    if failures and require_all:
        raise RuntimeError(f"Checksum contract failed for {root}: {failures[:20]}")
    return sums, failures

def verify_manifest_payload_seal(path):
    man = json.loads(Path(path).read_text(encoding="utf-8"))
    recorded = man.get("manifest_payload_sha256")
    if recorded is None:
        return man, None
    temp = dict(man)
    temp.pop("manifest_payload_sha256", None)
    observed = hashlib.sha256(canonical_json_bytes(temp)).hexdigest()
    if observed != recorded:
        raise RuntimeError(f"Manifest payload seal mismatch: {path}")
    return man, recorded

def safe_copy_verified(src, dst, expected_sha256=None):
    src = Path(src)
    dst = Path(dst)
    if not src.is_file():
        raise RuntimeError(f"Source artifact missing: {src}")
    observed = sha256_file(src)
    if expected_sha256 is not None and observed != expected_sha256:
        raise RuntimeError(
            f"Source hash mismatch before copy: {src}; expected={expected_sha256}; observed={observed}"
        )
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    copied = sha256_file(dst)
    if copied != observed:
        raise RuntimeError(f"Post-copy hash mismatch: {dst}")
    return observed

def hard_stop(message, details=None):
    payload = {"status": "HARD_STOP", "message": message, "details": details or {}, "timestamp_utc": utc_now()}
    if "RELEASE_ROOT" in globals():
        try:
            write_json(RELEASE_ROOT / "RELEASE_STATUS.json", payload)
        except Exception:
            pass
    raise RuntimeError(message)

def normalize_path_text(text):
    return str(text).replace("\\", "/")

def is_probable_text_file(path):
    return Path(path).suffix.lower() in {
        ".txt", ".md", ".json", ".csv", ".tsv", ".yaml", ".yml", ".py", ".ipynb", ".tex"
    }

def file_size(path):
    return int(Path(path).stat().st_size)


## 2. Search roots and empty release target

In [3]:

default_search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/mnt/data"), Path.cwd()]
extra = os.environ.get("SWARABYANJAN_RELEASE_SEARCH_ROOTS", "").strip()
if extra:
    default_search_roots.extend(Path(x) for x in extra.split(os.pathsep) if x.strip())

SEARCH_ROOTS = []
for p in default_search_roots:
    try:
        p = p.resolve()
    except Exception:
        p = Path(p)
    if p.exists() and p not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(p)

work_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RELEASE_ROOT = Path(os.environ.get(
    "SWARABYANJAN_RELEASE_OUT",
    str(work_root / "swarabyanjan-crossplatform-paper"),
)).resolve()

if RELEASE_ROOT.exists() and any(RELEASE_ROOT.iterdir()):
    hard_stop("Refusing to overwrite a non-empty release directory", {"release_root": str(RELEASE_ROOT)})
RELEASE_ROOT.mkdir(parents=True, exist_ok=True)

for rel in [
    "notebooks",
    "archive/historical_intermediate",
    *CANONICAL_STAGE_DIRS.values(),
    "figure_source_data",
    "manuscript_tables",
    "models_optional",
    "ANONYMIZED_RELEASE",
]:
    (RELEASE_ROOT / rel).mkdir(parents=True, exist_ok=True)

print("Search roots:", [str(x) for x in SEARCH_ROOTS])
print("Release root:", RELEASE_ROOT)


Search roots: ['/kaggle/input', '/kaggle/working']
Release root: /kaggle/working/swarabyanjan-crossplatform-paper


## 3. Canonical notebook discovery + automatic static contract scan

Notebook files are located by **filename plus SHA-256**, not filename alone. The scan extracts output-root declarations, file literals, manifest/checksum/status references, environment/hardware references, model/repository references, and dependency language from the actual canonical notebook source. It does not execute those notebooks.


In [4]:
# Frozen source identities derived once from the exact whole-file canonical notebooks whose
# SHA-256 values are declared in CANONICAL_NOTEBOOK_SHA256. This is a release-engineering
# identity layer only; it does not change any Stage 01-09 scientific identity.
#
# Deterministic CANONICAL_SOURCE rule:
#   1) parse the notebook JSON;
#   2) require exactly the frozen cell count and only code/markdown cells;
#   3) for every cell in order, retain exactly: {index, cell_type, source_text};
#      source_text is the exact concatenation of cells[*].source strings, with no whitespace
#      normalization;
#   4) JSON-serialize that ordered list with UTF-8, ensure_ascii=False, sort_keys=True,
#      separators=(",", ":"), allow_nan=False;
#   5) SHA-256 the resulting bytes.
# Outputs, execution_count, cell metadata, and notebook metadata are deliberately excluded.
CANONICAL_NOTEBOOK_SOURCE_SHA256 = {
    "01-dataset-acquisition-and-freeze.ipynb": "009f92a04bfd38d5f4b521fbbec3306845bafa3281206736b16b49e8f1514196",
    "02-p0-integrity-provenance-leakage-audit.ipynb": "2f248e31bdf6421a2c4fba6ac78d2300f1f479bd09da317d833f2f916179d68b",
    "03-primary-model-development.ipynb": "260246bd57233e6a663ea5258a69e8a500b5b1b4a059381df51125b148a0271f",
    "04-calibration.ipynb": "d5f0c82ca353780e15cef5f27b8ae5fbcb3e840a18624ef1c7196b8e821e9908",
    "05-external-validation-and-h3-vfinal.ipynb": "28f131f3004bd38a057a0f17f0a89e14fd6fbd13c5f5d17f1a540898736b869b",
    "06-fixed-budget-reliability-evaluation-vfinal.ipynb": "3edac9c63c3cb5711ee5a4e01afff89727929fd1629afab1256136335b3bb809",
    "07-h3-h4-bridge-exploratory-vfinal.ipynb": "c5930a6a11ffdb49f169e75b8dff73f7fbdceed5eb5da160646200c5164efa87",
    "08-manuscript-figures-and-tables-vfinal.ipynb": "038699569e4562388ebd77df3a8cb6069d0d0a37040b839222a21cde8b795042",
    "09-source-publisher-cluster-sensitivity-vfinal.ipynb": "e214251831c1dde32d60ffe0e06196e331e05ce9d331747861fc058751b39a65",
}
CANONICAL_NOTEBOOK_CELL_COUNTS = {
    "01-dataset-acquisition-and-freeze.ipynb": 19,
    "02-p0-integrity-provenance-leakage-audit.ipynb": 23,
    "03-primary-model-development.ipynb": 22,
    "04-calibration.ipynb": 14,
    "05-external-validation-and-h3-vfinal.ipynb": 19,
    "06-fixed-budget-reliability-evaluation-vfinal.ipynb": 14,
    "07-h3-h4-bridge-exploratory-vfinal.ipynb": 13,
    "08-manuscript-figures-and-tables-vfinal.ipynb": 16,
    "09-source-publisher-cluster-sensitivity-vfinal.ipynb": 12,
}


def find_files_named(name):
    # Exact-name lookup is retained for non-notebook stage artifacts and historical handling.
    found = []
    for root in SEARCH_ROOTS:
        try:
            found.extend(root.rglob(name))
        except (PermissionError, OSError):
            continue
    return sorted({p.resolve() for p in found if p.is_file()}, key=str)


def find_all_accessible_ipynb():
    """Recursively enumerate every directly mounted .ipynb under all allowed search roots."""
    found = []
    for root in SEARCH_ROOTS:
        try:
            found.extend(root.rglob("*.ipynb"))
        except (PermissionError, OSError):
            continue
    return sorted({p.resolve() for p in found if p.is_file()}, key=str)


def locate_file_by_hash(name, expected_sha256):
    matches = []
    for p in find_files_named(name):
        try:
            if sha256_file(p) == expected_sha256:
                matches.append(p)
        except OSError:
            continue
    if not matches:
        hard_stop("Hash-gated notebook/file not found",
                  {"name": name, "expected_sha256": expected_sha256, "search_roots": [str(x) for x in SEARCH_ROOTS]})
    return matches[0], matches


def _notebook_source_text(cell):
    source = cell.get("source", [])
    if isinstance(source, str):
        return source
    if isinstance(source, list) and all(isinstance(x, str) for x in source):
        return "".join(source)
    raise ValueError("Notebook cell source is neither a string nor a list of strings")


def canonical_source_sha256_from_notebook_obj(obj):
    cells = obj.get("cells")
    if not isinstance(cells, list):
        raise ValueError("Notebook has no valid cells list")
    ordered = []
    for index, cell in enumerate(cells):
        if not isinstance(cell, dict):
            raise ValueError(f"Notebook cell {index} is not an object")
        cell_type = cell.get("cell_type")
        if cell_type not in {"code", "markdown"}:
            raise ValueError(f"Unsupported canonical cell type at index {index}: {cell_type!r}")
        ordered.append({
            "index": index,
            "cell_type": cell_type,
            "source": _notebook_source_text(cell),
        })
    payload = json.dumps(
        ordered, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def notebook_execution_state(obj):
    code_cells = [c for c in obj.get("cells", []) if c.get("cell_type") == "code"]
    populated_counts = sum(c.get("execution_count") is not None for c in code_cells)
    output_objects = sum(len(c.get("outputs", [])) for c in code_cells)
    if populated_counts == 0 and output_objects == 0:
        state = "UNEXECUTED"
    elif populated_counts == len(code_cells):
        state = "EXECUTED"
    else:
        state = "PARTIALLY_EXECUTED"
    return {
        "state": state,
        "code_cell_count": len(code_cells),
        "populated_execution_count_cells": populated_counts,
        "output_object_count": output_objects,
    }


def inspect_notebook_bytes(data, expected_cell_count=None):
    whole_sha = hashlib.sha256(data).hexdigest()
    try:
        obj = json.loads(data.decode("utf-8"))
    except Exception as e:
        return {
            "parse_ok": False,
            "whole_notebook_sha256": whole_sha,
            "canonical_source_sha256": None,
            "cell_count": None,
            "execution_state": None,
            "parse_error": f"{type(e).__name__}: {e}",
        }
    try:
        source_sha = canonical_source_sha256_from_notebook_obj(obj)
        source_error = None
    except Exception as e:
        source_sha = None
        source_error = f"{type(e).__name__}: {e}"
    cell_count = len(obj.get("cells", [])) if isinstance(obj.get("cells"), list) else None
    return {
        "parse_ok": True,
        "whole_notebook_sha256": whole_sha,
        "canonical_source_sha256": source_sha,
        "cell_count": cell_count,
        "expected_cell_count_match": expected_cell_count is None or cell_count == expected_cell_count,
        "execution_state": notebook_execution_state(obj),
        "source_identity_error": source_error,
    }


def inspect_notebook_path(path, expected_cell_count=None):
    return inspect_notebook_bytes(Path(path).read_bytes(), expected_cell_count=expected_cell_count)


def _all_zip_candidates():
    found = []
    for root in SEARCH_ROOTS:
        try:
            found.extend(root.rglob("*.zip"))
        except (PermissionError, OSError):
            continue
    return sorted({p.resolve() for p in found if p.is_file()}, key=str)


def _candidate_rejection_reason(identity, expected_whole, expected_source, expected_count):
    # Authentication precedence is exact frozen whole-file SHA first.
    if identity.get("whole_notebook_sha256") == expected_whole:
        return "ACCEPT_WHOLE_FILE"
    if not identity.get("parse_ok"):
        return "NOTEBOOK_PARSE_FAILED"
    if identity.get("cell_count") != expected_count:
        return "CELL_COUNT_MISMATCH"
    if identity.get("canonical_source_sha256") != expected_source:
        return "CANONICAL_SOURCE_SHA256_MISMATCH"
    return "ACCEPT_CANONICAL_SOURCE"


def collect_canonical_notebook_candidates(
    name, expected_whole, expected_source, expected_count,
    direct_notebook_pool, zip_candidates=None,
):
    """
    Build candidates without using notebook filenames as identity.
    Direct mounted .ipynb files are primary. ZIP members are optional secondary candidates
    and are considered only when explicitly supplied by the caller.
    """
    candidates = []

    for base in direct_notebook_pool:
        c = dict(base)
        c["reason_for_rejection"] = _candidate_rejection_reason(
            c, expected_whole, expected_source, expected_count
        )
        candidates.append(c)

    for zp in (zip_candidates or []):
        try:
            with zipfile.ZipFile(zp, "r") as zf:
                # No filename matching: inspect every notebook member, including nested paths.
                members = sorted(
                    m for m in zf.namelist()
                    if not m.endswith("/") and m.lower().endswith(".ipynb")
                )
                for member in members:
                    try:
                        data = zf.read(member)
                        ident = inspect_notebook_bytes(data, expected_cell_count=expected_count)
                        c = {
                            "kind": "ZIP_MEMBER",
                            "candidate_path": f"zip://{zp}!/{member}",
                            "path": None,
                            "archive": zp,
                            "member": member,
                            **ident,
                        }
                    except Exception as e:
                        c = {
                            "kind": "ZIP_MEMBER",
                            "candidate_path": f"zip://{zp}!/{member}",
                            "path": None,
                            "archive": zp,
                            "member": member,
                            "parse_ok": False,
                            "whole_notebook_sha256": None,
                            "canonical_source_sha256": None,
                            "cell_count": None,
                            "execution_state": None,
                            "inspection_error": f"{type(e).__name__}: {e}",
                        }
                    c["reason_for_rejection"] = _candidate_rejection_reason(
                        c, expected_whole, expected_source, expected_count
                    )
                    candidates.append(c)
        except (zipfile.BadZipFile, PermissionError, OSError):
            continue

    # Stable de-duplication by direct path or archive+member identity.
    dedup = {}
    for c in candidates:
        dedup[c["candidate_path"]] = c
    return [dedup[k] for k in sorted(dedup)]

def _candidate_mount_provenance(candidate):
    """
    Deterministic provenance classification used ONLY AFTER cryptographic notebook
    authentication. It never establishes canonical identity.

    STAGE_ROOT_NOTEBOOK means a directly mounted file named ``__notebook__.ipynb``
    located either directly under a configured search root or exactly one directory
    below that root (the Kaggle Notebook-input form is normally
    ``/kaggle/input/<mounted-stage>/__notebook__.ipynb``). Deeper paths are treated
    as nested result/artifact copies.

    When search roots overlap, classification uses the broadest containing search
    root so that a nested cwd cannot make a deeper artifact copy appear stage-root.
    """
    if candidate.get("kind") != "DIRECT_FILE" or candidate.get("path") is None:
        return {
            "provenance_class": "ARCHIVE_OR_NON_DIRECT",
            "is_stage_root_notebook": False,
            "search_root": None,
            "relative_depth": None,
        }

    path = Path(candidate["path"]).resolve()
    containing = []
    for root in SEARCH_ROOTS:
        try:
            root_resolved = Path(root).resolve()
            rel = path.relative_to(root_resolved)
            containing.append((len(root_resolved.parts), str(root_resolved), rel))
        except Exception:
            continue

    if not containing:
        return {
            "provenance_class": "DIRECT_OUTSIDE_SEARCH_ROOT",
            "is_stage_root_notebook": False,
            "search_root": None,
            "relative_depth": None,
        }

    # Fewest root path components = broadest containing configured search root.
    _, root_text, rel = sorted(containing, key=lambda x: (x[0], x[1]))[0]
    rel_depth = len(rel.parts)
    is_stage_root = (
        path.name == "__notebook__.ipynb"
        and rel_depth in {1, 2}
    )
    return {
        "provenance_class": (
            "STAGE_ROOT_NOTEBOOK" if is_stage_root else "NESTED_DIRECT_NOTEBOOK_COPY"
        ),
        "is_stage_root_notebook": is_stage_root,
        "search_root": root_text,
        "relative_depth": rel_depth,
    }


def _authenticated_candidate_fallback_sort_key(candidate):
    """
    Stable path-provenance ordering used only after an identity tier has already
    authenticated every candidate in the tied set. Direct files outrank archive
    members, then shallower mounted paths outrank deeper paths, then candidate_path
    supplies deterministic ordering. This ranking never substitutes for SHA identity.
    """
    provenance = _candidate_mount_provenance(candidate)
    direct_rank = 0 if candidate.get("kind") == "DIRECT_FILE" else 1
    depth = provenance["relative_depth"]
    depth_rank = depth if isinstance(depth, int) else 10**9
    return (direct_rank, depth_rank, str(candidate.get("candidate_path", "")))


def _resolve_authenticated_duplicate_candidates(matches, identity_tier):
    """
    Resolve duplicate filesystem copies that have ALREADY passed the same frozen
    identity tier. A unique directly mounted stage-root ``__notebook__.ipynb`` wins
    over nested result/artifact copies. More than one authenticated stage-root copy
    is the only unresolved provenance ambiguity and therefore hard-stops.

    If no stage-root copy exists, deterministic mounted-path provenance ordering is
    used among the already cryptographically identical candidates. This changes only
    which byte wrapper is copied; it does not alter canonical source authentication.
    """
    if len(matches) == 1:
        return matches[0], None

    stage_root_matches = [
        c for c in matches
        if _candidate_mount_provenance(c)["is_stage_root_notebook"]
    ]

    if len(stage_root_matches) == 1:
        selected = dict(stage_root_matches[0])
        selected["selection_tiebreaker"] = {
            "rule": "PREFER_UNIQUE_AUTHENTICATED_STAGE_ROOT___notebook__.ipynb",
            "identity_tier": identity_tier,
            "authenticated_match_count": len(matches),
            "stage_root_match_count": 1,
            "ignored_authenticated_duplicate_candidates": [
                c["candidate_path"] for c in matches
                if c["candidate_path"] != selected["candidate_path"]
            ],
        }
        return selected, None

    if len(stage_root_matches) > 1:
        return None, {
            "reason": f"AMBIGUOUS_MULTIPLE_STAGE_ROOT_{identity_tier}_MATCHES",
            "matching_candidate_paths": [c["candidate_path"] for c in matches],
            "stage_root_candidate_paths": [c["candidate_path"] for c in stage_root_matches],
        }

    selected = dict(sorted(matches, key=_authenticated_candidate_fallback_sort_key)[0])
    selected["selection_tiebreaker"] = {
        "rule": "NO_STAGE_ROOT_PRESENT_DETERMINISTIC_AUTHENTICATED_FALLBACK",
        "identity_tier": identity_tier,
        "authenticated_match_count": len(matches),
        "stage_root_match_count": 0,
        "ordered_authenticated_candidates": [
            c["candidate_path"]
            for c in sorted(matches, key=_authenticated_candidate_fallback_sort_key)
        ],
    }
    return selected, None


def select_canonical_notebook_candidate(name, candidates, expected_whole, expected_source, expected_count):
    # LEVEL A: exact frozen whole-file SHA, regardless of filename. Highest priority.
    whole_matches = [
        c for c in candidates
        if c.get("whole_notebook_sha256") == expected_whole
    ]
    if whole_matches:
        selected, selection_error = _resolve_authenticated_duplicate_candidates(
            whole_matches, "WHOLE_FILE"
        )
        if selected is not None:
            return selected, "WHOLE_FILE", None
        return None, None, selection_error

    # LEVEL B: exact frozen canonical-source SHA + frozen cell count.
    # Filename/path is NOT identity. Provenance path is consulted only after every
    # candidate below has already satisfied both frozen source-authentication gates.
    source_matches = [
        c for c in candidates
        if c.get("parse_ok")
        and c.get("cell_count") == expected_count
        and c.get("canonical_source_sha256") == expected_source
    ]
    if source_matches:
        selected, selection_error = _resolve_authenticated_duplicate_candidates(
            source_matches, "CANONICAL_SOURCE"
        )
        if selected is not None:
            return selected, "CANONICAL_SOURCE", None
        return None, None, selection_error

    return None, None, {"reason": "NO_VALID_CANONICAL_NOTEBOOK_MATCH"}

def materialize_selected_notebook(name, selected):
    if selected["kind"] == "DIRECT_FILE":
        return Path(selected["path"]).resolve()
    staging = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()) / ".stage10_notebook_inputs"
    staging.mkdir(parents=True, exist_ok=True)
    target = staging / name
    with zipfile.ZipFile(selected["archive"], "r") as zf:
        data = zf.read(selected["member"])
    target.write_bytes(data)
    observed = sha256_file(target)
    if observed != selected["whole_notebook_sha256"]:
        hard_stop("Notebook archive extraction hash mismatch", {
            "archive": str(selected["archive"]),
            "member": selected["member"],
            "expected_extracted_sha256": selected["whole_notebook_sha256"],
            "observed_extracted_sha256": observed,
        })
    return target.resolve()


def _diagnostic_candidate_view(c):
    provenance = _candidate_mount_provenance(c)
    return {
        "candidate_path": c.get("candidate_path"),
        "candidate_whole_file_sha256": c.get("whole_notebook_sha256"),
        "candidate_source_sha256": c.get("canonical_source_sha256"),
        "cell_count": c.get("cell_count"),
        "execution_state": c.get("execution_state"),
        "reason_for_rejection": c.get("reason_for_rejection"),
        "provenance_class": provenance["provenance_class"],
        "is_stage_root_notebook": provenance["is_stage_root_notebook"],
        "provenance_search_root": provenance["search_root"],
        "provenance_relative_depth": provenance["relative_depth"],
        "parse_error": c.get("parse_error") or c.get("source_identity_error") or c.get("inspection_error"),
    }


def locate_historical_notebook_by_hash(name, expected_sha256, zip_candidates):
    """Locate an optional historical notebook only by its exact frozen whole-file SHA-256.

    Historical/intermediate notebooks are archival provenance, not runtime requirements for
    scientific reproduction. Filename matches are only search hints and never establish identity.
    """
    direct = []
    for p in find_files_named(name):
        try:
            if sha256_file(p) == expected_sha256:
                direct.append(p)
        except OSError:
            continue
    if direct:
        direct = sorted(set(direct), key=str)
        return direct[0], direct

    matches = []
    staging = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()) / ".stage10_notebook_inputs"
    staging.mkdir(parents=True, exist_ok=True)
    for zp in zip_candidates:
        try:
            with zipfile.ZipFile(zp, "r") as zf:
                members = sorted(m for m in zf.namelist() if not m.endswith("/") and Path(m).name == name)
                for member in members:
                    data = zf.read(member)
                    if hashlib.sha256(data).hexdigest() != expected_sha256:
                        continue
                    target = staging / name
                    target.write_bytes(data)
                    if sha256_file(target) != expected_sha256:
                        hard_stop("Historical notebook archive extraction hash mismatch", {
                            "archive": str(zp), "member": member
                        })
                    matches.append(target.resolve())
        except (zipfile.BadZipFile, PermissionError, OSError):
            continue
    matches = sorted(set(matches), key=str)
    if matches:
        return matches[0], matches
    return None, []


def notebook_source_text(path):
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    return "\n".join(_notebook_source_text(cell) for cell in obj.get("cells", []))


def notebook_code_text(path):
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    return "\n".join(
        _notebook_source_text(cell)
        for cell in obj.get("cells", [])
        if cell.get("cell_type") == "code"
    )


def static_contract_scan(path):
    src = notebook_source_text(path)
    file_literals = sorted(set(re.findall(
        r"[\"']([^\"']+\.(?:json|csv|parquet|txt|md|pkl|joblib|pt|bin|zip|yaml|yml|png|pdf|svg|tex|npy|npz))[\"']",
        src, flags=re.I
    )))
    output_root_lines = [
        line.strip() for line in src.splitlines()
        if re.search(r"\b(?:OUT_ROOT|OUTPUT_PARENT|STAGE_RESULTS_ROOT|RUN_DIR|ROOT)\s*=", line)
    ]
    identity_lines = [
        line.strip() for line in src.splitlines()
        if any(token in line for token in [
            "PROTOCOL_VERSION", "CANONICAL_VERSION", "BRIDGE_VERSION", "SENSITIVITY_VERSION",
            "EXPECTED_ZIP_SHA256", "EXPECTED_FROZEN_ZIP_SHA256"
        ])
    ]
    env_lines = [
        line.strip() for line in src.splitlines()
        if re.search(r"(?i)\b(python|platform|packages|environment|gpu|cpu|cuda|accelerator)\b", line)
    ]
    dataset_hash_lines = [
        line.strip() for line in src.splitlines()
        if re.search(r"(?i)\b(?:sha256|git_blob_sha1|revision|commit|doi)\b", line)
        and re.search(r"(?i)\b(?:dataset|banglabait|banglaclick|banmani|banfakenews|baitbuster|freeze|source)\b", line)
    ]
    model_tokenizer_reference_lines = [
        line.strip() for line in src.splitlines()
        if re.search(r"(?i)\b(?:model|tokenizer|from_pretrained|save_pretrained|repo_id|hf_models|banglabert|xlm-roberta|xlmr)\b", line)
    ]
    dependency_lines = [
        line.strip() for line in src.splitlines()
        if re.search(r"(?i)\b(upstream|downstream|manifest_sha256|checksum_sha256|prediction_sha256)\b", line)
    ]
    return {
        "notebook": Path(path).name,
        "sha256": sha256_file(path),
        "file_literals": file_literals,
        "output_root_declarations": output_root_lines,
        "identity_declarations": identity_lines,
        "environment_hardware_declarations": env_lines,
        "dataset_hash_provenance_declarations": dataset_hash_lines,
        "model_tokenizer_reference_declarations": model_tokenizer_reference_lines,
        "dependency_declarations": dependency_lines,
    }


# Canonical notebook discovery is direct-file first and filename-independent.
# Kaggle "Notebook" inputs expose saved output files; Stage 10 must not assume that the
# source .ipynb itself is mounted. We therefore enumerate every accessible .ipynb and
# authenticate only by the frozen identity contract.
DIRECT_NOTEBOOK_FILES = find_all_accessible_ipynb()
DIRECT_NOTEBOOK_POOL = []
for p in DIRECT_NOTEBOOK_FILES:
    try:
        ident = inspect_notebook_path(p)
        DIRECT_NOTEBOOK_POOL.append({
            "kind": "DIRECT_FILE",
            "candidate_path": str(p),
            "path": p,
            "archive": None,
            "member": None,
            **ident,
        })
    except Exception as e:
        DIRECT_NOTEBOOK_POOL.append({
            "kind": "DIRECT_FILE",
            "candidate_path": str(p),
            "path": p,
            "archive": None,
            "member": None,
            "parse_ok": False,
            "whole_notebook_sha256": None,
            "canonical_source_sha256": None,
            "cell_count": None,
            "execution_state": None,
            "inspection_error": f"{type(e).__name__}: {e}",
        })

ZIP_CANDIDATES = _all_zip_candidates()  # retained only as a non-primary compatibility path
NOTEBOOK_PATHS = {}
NOTEBOOK_ALIASES = {}
NOTEBOOK_IDENTITY_AUDIT = {}

def _is_under(path, root):
    try:
        Path(path).resolve().relative_to(Path(root).resolve())
        return True
    except Exception:
        return False

input_root = Path("/kaggle/input")
ipynb_under_kaggle_input = [
    str(p) for p in DIRECT_NOTEBOOK_FILES
    if input_root.exists() and _is_under(p, input_root)
]

# First attempt: DIRECT mounted .ipynb files only, irrespective of filename.
direct_selection = {}
direct_reports = {}
for name in CANONICAL_NOTEBOOKS:
    expected_whole = CANONICAL_NOTEBOOK_SHA256[name]
    expected_source = CANONICAL_NOTEBOOK_SOURCE_SHA256[name]
    expected_count = CANONICAL_NOTEBOOK_CELL_COUNTS[name]
    candidates = collect_canonical_notebook_candidates(
        name, expected_whole, expected_source, expected_count,
        DIRECT_NOTEBOOK_POOL, zip_candidates=None,
    )
    selected, match_type, selection_error = select_canonical_notebook_candidate(
        name, candidates, expected_whole, expected_source, expected_count
    )
    direct_selection[name] = (selected, match_type, selection_error)
    direct_reports[name] = {
        "name": name,
        "expected_whole_file_sha256": expected_whole,
        "expected_canonical_source_sha256": expected_source,
        "expected_cell_count": expected_count,
        "candidate_paths_found": [c["candidate_path"] for c in candidates],
        "candidates": [_diagnostic_candidate_view(c) for c in candidates],
        "selection_error": selection_error,
    }

direct_complete = all(direct_selection[name][0] is not None for name in CANONICAL_NOTEBOOKS)
direct_ambiguities = {
    name: err for name, (_, _, err) in direct_selection.items()
    if err and str(err.get("reason", "")).startswith("AMBIGUOUS_")
}

# Ambiguity is always fatal: never choose between multiple authenticated direct candidates.
if direct_ambiguities:
    ambiguity_report = {
        "status": "FAIL",
        "diagnosis": "Multiple direct notebook files satisfy a frozen canonical identity; refusing to guess.",
        "search_roots": [str(x) for x in SEARCH_ROOTS],
        "all_direct_ipynb_files_found": [str(p) for p in DIRECT_NOTEBOOK_FILES],
        "canonical_notebooks": direct_reports,
        "ambiguities": direct_ambiguities,
    }
    diagnostic_path = RELEASE_ROOT / "NOTEBOOK_DISCOVERY_DIAGNOSTIC.json"
    write_json(diagnostic_path, ambiguity_report)
    print("NOTEBOOK DISCOVERY AMBIGUITY DIAGNOSTIC")
    print(json.dumps(ambiguity_report, indent=2, sort_keys=True, ensure_ascii=False))
    hard_stop(
        "Ambiguous canonical notebook identity among directly mounted .ipynb files",
        {"diagnostic_report": str(diagnostic_path)},
    )

# Kaggle NOTEBOOK-input visibility gate.
# If the nine canonical source notebooks are not directly visible/authenticatable as .ipynb
# files, do not silently substitute ZIP members or output artifacts.
if input_root.exists() and not direct_complete:
    if not ipynb_under_kaggle_input:
        diagnosis = (
            "No source .ipynb files are exposed under /kaggle/input. Kaggle Notebook inputs "
            "mount saved notebook OUTPUT files, not the notebook source itself unless that source "
            "was explicitly emitted as an output artifact. The canonical notebook sources therefore "
            "cannot be authenticated from the currently mounted Notebook inputs."
        )
        visibility_status = "KAGGLE_NOTEBOOK_OUTPUTS_ONLY_NO_SOURCE_IPYNB_VISIBLE"
    else:
        diagnosis = (
            "Direct .ipynb files are present under /kaggle/input, but the nine canonical notebook "
            "sources do not all satisfy their frozen whole-file or canonical-source identities. "
            "Stage 10 will not fall back to filenames, ZIP members, or output artifacts on Kaggle."
        )
        visibility_status = "DIRECT_IPYNB_PRESENT_BUT_CANONICAL_IDENTITIES_INCOMPLETE"

    visibility = {
        "status": "FAIL",
        "visibility_status": visibility_status,
        "diagnosis": diagnosis,
        "search_roots": [str(x) for x in SEARCH_ROOTS],
        "all_direct_ipynb_files_found": [str(p) for p in DIRECT_NOTEBOOK_FILES],
        "direct_ipynb_files_under_kaggle_input": ipynb_under_kaggle_input,
        "canonical_notebooks": direct_reports,
        "zip_candidates_ignored_for_this_gate": [str(x) for x in ZIP_CANDIDATES],
        "required_action": (
            "Provide the exact canonical notebook source .ipynb files as directly mounted "
            "filesystem inputs. Notebook-output mounts alone are insufficient for source authentication."
        ),
    }
    diagnostic_path = RELEASE_ROOT / "KAGGLE_NOTEBOOK_SOURCE_VISIBILITY_DIAGNOSTIC.json"
    write_json(diagnostic_path, visibility)
    print("KAGGLE NOTEBOOK SOURCE VISIBILITY DIAGNOSTIC")
    print(json.dumps(visibility, indent=2, sort_keys=True, ensure_ascii=False))
    hard_stop(
        "Canonical notebook source .ipynb files are not fully exposed/authenticated by the mounted Kaggle inputs",
        {"diagnostic_report": str(diagnostic_path)},
    )

# Outside Kaggle, retain compact-ZIP compatibility only as a secondary path after recursive
# direct .ipynb discovery has failed. It is never the primary Kaggle mechanism.
use_zip_fallback = not direct_complete and not input_root.exists()

NOTEBOOK_DISCOVERY_DIAGNOSTIC = {
    "search_roots": [str(x) for x in SEARCH_ROOTS],
    "direct_ipynb_candidates_found": [str(p) for p in DIRECT_NOTEBOOK_FILES],
    "zip_candidates_found": [str(x) for x in ZIP_CANDIDATES],
    "zip_fallback_used": use_zip_fallback,
    "canonical_notebooks": {},
}
_discovery_failures = []

for name in CANONICAL_NOTEBOOKS:
    expected_whole = CANONICAL_NOTEBOOK_SHA256[name]
    expected_source = CANONICAL_NOTEBOOK_SOURCE_SHA256[name]
    expected_count = CANONICAL_NOTEBOOK_CELL_COUNTS[name]

    if direct_complete:
        candidates = collect_canonical_notebook_candidates(
            name, expected_whole, expected_source, expected_count,
            DIRECT_NOTEBOOK_POOL, zip_candidates=None,
        )
    else:
        candidates = collect_canonical_notebook_candidates(
            name, expected_whole, expected_source, expected_count,
            DIRECT_NOTEBOOK_POOL, zip_candidates=ZIP_CANDIDATES if use_zip_fallback else None,
        )

    selected, match_type, selection_error = select_canonical_notebook_candidate(
        name, candidates, expected_whole, expected_source, expected_count
    )
    report = {
        "name": name,
        "expected_whole_file_sha256": expected_whole,
        "expected_canonical_source_sha256": expected_source,
        "expected_cell_count": expected_count,
        "candidate_paths_found": [c["candidate_path"] for c in candidates],
        "candidates": [_diagnostic_candidate_view(c) for c in candidates],
        "selection_error": selection_error,
    }
    if selected is None:
        _discovery_failures.append({"name": name, **(selection_error or {})})
        NOTEBOOK_DISCOVERY_DIAGNOSTIC["canonical_notebooks"][name] = report
        continue

    path = materialize_selected_notebook(name, selected)
    NOTEBOOK_PATHS[name] = path
    # Exactly one authenticated canonical candidate is allowed.
    NOTEBOOK_ALIASES[name] = [selected["candidate_path"]]
    selected_record = {
        "name": name,
        "selected_path": str(path),
        "selected_candidate": selected["candidate_path"],
        "expected_whole_file_sha256": expected_whole,
        "whole_notebook_sha256": selected["whole_notebook_sha256"],
        "canonical_source_sha256": selected["canonical_source_sha256"],
        "expected_cell_count": expected_count,
        "cell_count": selected["cell_count"],
        "execution_state": selected["execution_state"],
        "match_type": match_type,
        "selection_tiebreaker": selected.get("selection_tiebreaker"),
    }
    NOTEBOOK_IDENTITY_AUDIT[name] = selected_record
    report.update({"selected": selected_record})
    NOTEBOOK_DISCOVERY_DIAGNOSTIC["canonical_notebooks"][name] = report

if _discovery_failures:
    diagnostic_path = RELEASE_ROOT / "NOTEBOOK_DISCOVERY_DIAGNOSTIC.json"
    write_json(diagnostic_path, NOTEBOOK_DISCOVERY_DIAGNOSTIC)
    print("NOTEBOOK DISCOVERY FAILURE DIAGNOSTIC")
    print(json.dumps(NOTEBOOK_DISCOVERY_DIAGNOSTIC, indent=2, sort_keys=True, ensure_ascii=False))
    hard_stop("Canonical notebook discovery/identity gate failed", {
        "failures": _discovery_failures,
        "diagnostic_report": str(diagnostic_path),
    })

# Persist the successful identity audit. This reports whole-file and source identities separately.
write_json(RELEASE_ROOT / "NOTEBOOK_IDENTITY_AUDIT.json", NOTEBOOK_IDENTITY_AUDIT)


HISTORICAL_NOTEBOOK_PATHS = {}
HISTORICAL_NOTEBOOK_AUDIT = {}
for name in HISTORICAL_NOTEBOOKS:
    p, matches = locate_historical_notebook_by_hash(name, HISTORICAL_NOTEBOOK_SHA256[name], ZIP_CANDIDATES)
    if p is None:
        HISTORICAL_NOTEBOOK_AUDIT[name] = {
            "name": name,
            "expected_whole_file_sha256": HISTORICAL_NOTEBOOK_SHA256[name],
            "present": False,
            "availability": "NOT_MOUNTED_IN_CURRENT_RUNTIME",
            "match_type": None,
            "whole_notebook_sha256": None,
            "matching_candidate_count": 0,
        }
    else:
        HISTORICAL_NOTEBOOK_PATHS[name] = p
        HISTORICAL_NOTEBOOK_AUDIT[name] = {
            "name": name,
            "expected_whole_file_sha256": HISTORICAL_NOTEBOOK_SHA256[name],
            "present": True,
            "availability": "MOUNTED_AND_VERIFIED",
            "match_type": "WHOLE_FILE",
            "whole_notebook_sha256": HISTORICAL_NOTEBOOK_SHA256[name],
            "matching_candidate_count": len(matches),
        }

NOTEBOOK_STATIC_SCAN = {
    f"{i:02d}": static_contract_scan(NOTEBOOK_PATHS[name])
    for i, name in enumerate(CANONICAL_NOTEBOOKS, start=1)
}
write_json(RELEASE_ROOT / "NOTEBOOK_STATIC_CONTRACT_SCAN.json", NOTEBOOK_STATIC_SCAN)

if len(NOTEBOOK_PATHS) != 9:
    hard_stop("Canonical notebook cardinality violation", {
        "canonical_count": len(NOTEBOOK_PATHS),
    })

print("Canonical notebook identity gate: PASS (9/9)")
for name in CANONICAL_NOTEBOOKS:
    rec = NOTEBOOK_IDENTITY_AUDIT[name]
    print(
        f"  {name}: match_type={rec['match_type']} "
        f"whole_notebook_sha256={rec['whole_notebook_sha256']} "
        f"canonical_source_sha256={rec['canonical_source_sha256']} "
        f"execution_state={rec['execution_state']['state']}"
    )
present_historical = sum(1 for rec in HISTORICAL_NOTEBOOK_AUDIT.values() if rec["present"])
print(
    f"Historical notebook identification: optional archive inputs present={present_historical}/3 — "
    "HISTORICAL / NON-CANONICAL — DO NOT EXECUTE FOR REPRODUCTION"
)


Canonical notebook identity gate: PASS (9/9)
  01-dataset-acquisition-and-freeze.ipynb: match_type=CANONICAL_SOURCE whole_notebook_sha256=0223500a7d1a58c1044d664e01c20e7e776fd4ceb58e8e8575181ac1944c0481 canonical_source_sha256=009f92a04bfd38d5f4b521fbbec3306845bafa3281206736b16b49e8f1514196 execution_state=EXECUTED
  02-p0-integrity-provenance-leakage-audit.ipynb: match_type=CANONICAL_SOURCE whole_notebook_sha256=77f5ad63c6c60dc637d15e2f7eec544e23138fb8f7d3b9e1c25e083f7f601d22 canonical_source_sha256=2f248e31bdf6421a2c4fba6ac78d2300f1f479bd09da317d833f2f916179d68b execution_state=EXECUTED
  03-primary-model-development.ipynb: match_type=CANONICAL_SOURCE whole_notebook_sha256=9847f4b0d6f31afca35a698557f9bf3cf339f37b9c14386829960474f59645da canonical_source_sha256=260246bd57233e6a663ea5258a69e8a500b5b1b4a059381df51125b148a0271f execution_state=EXECUTED
  04-calibration.ipynb: match_type=CANONICAL_SOURCE whole_notebook_sha256=64d23570c8df9c0559fce2b4194f40f3f298b9e9b2ca844644382f5e0bf5f4e

## 4. Explicit artifact contracts and dependency map

In [5]:

ARTIFACT_CONTRACTS = {
    "01": {
        "notebook": CANONICAL_NOTEBOOKS[0],
        "output_root_hint": "EXTERNAL-DATASET-BYTES-v1",
        "control_artifacts": [
            "ACQUISITION_MANIFEST.json", "ACQUISITION_STATUS.json", "SOURCES_LOCK.json",
            "SHA256SUMS.txt", "README_HANDOFF.txt",
        ],
        "required_data_interfaces": ["BanglaBait/repository_snapshot/test.csv"],
        "downstream": ["02", "03", "09"],
    },
    "02": {
        "notebook": CANONICAL_NOTEBOOKS[1],
        "output_root_hint": "03_results/02_p0_integrity_provenance_leakage_audit/<execution_id>",
        "control_artifacts": [
            "P0_AUDIT_REPORT.json", "P0_AUDIT_SUMMARY.md", "P0_BLOCKER_FORENSICS.json",
            "PROVENANCE_DEPENDENCY_GRAPH.json", "SPLIT_FIREWALL.json",
            "DATASET_SCHEMA_REPORT.csv", "LABEL_DISTRIBUTION_REPORT.csv", "DUPLICATE_REPORT.csv",
            "CROSS_DATASET_OVERLAP_REPORT.csv", "CONSTRUCT_ALIGNMENT_MATRIX.csv",
            "PROTECTED_TITLE_QUALITY_REPORT.csv", "P0_HASH_VERIFICATION.txt",
            "P0_RUN_METADATA.json", "EXECUTED_NOTEBOOK_CAPTURE_STATUS.json",
            "P0_DETERMINISTIC_OUTPUT_SHA256SUMS.txt", "P0_OUTPUT_SHA256SUMS.txt",
        ],
        "required_data_interfaces": [],
        "downstream": ["03"],
    },
    "03": {
        "notebook": CANONICAL_NOTEBOOKS[2],
        "output_root_hint": "03_results/03_primary_model_development",
        "control_artifacts": [
            "ENVIRONMENT_PREFLIGHT.json", "INPUT_IDENTITY.json", "FROZEN_INPUT_DISCOVERY.json",
            "CLEAN_DATA_MANIFEST.json", "CLEAN_ROW_MANIFEST.parquet", "CLEAN_SPLIT_SUMMARY.csv",
            "D0_REPORT.json", "DECONTAMINATION_FORENSICS.json", "CUE_FEATURE_SCHEMA.json",
            "TRAIN_DEV_SPLIT_ROWS.parquet", "TRAIN_DEV_SPLIT_MANIFEST.json",
            "MODEL_CONFIG_LOCK.json", "MODEL_ARTIFACT_MANIFEST.json", "RUN_METADATA.json",
            "STAGE03_SUMMARY.md", "STAGE03_SHA256SUMS.txt", "STAGE_STATUS.json",
        ],
        "required_data_interfaces": [
            "clean_splits/banglabait_train_clean.parquet",
            "clean_splits/banglabait_dev_clean.parquet",
            "clean_splits/banglabait_test_clean.parquet",
        ],
        "downstream": ["04", "05", "09"],
    },
    "04": {
        "notebook": CANONICAL_NOTEBOOKS[3],
        "output_root_hint": "03_results/04_calibration",
        "control_artifacts": [
            "ENVIRONMENT_PREFLIGHT.json", "CALIBRATION_REPORT.json", "TEMPERATURES.json",
            "CALIBRATION_ARTIFACT_MANIFEST.json", "DEV_PREDICTIONS.parquet",
            "RUN_METADATA.json", "STAGE04_SUMMARY.md", "STAGE04_SHA256SUMS.txt", "STAGE_STATUS.json",
        ],
        "required_data_interfaces": [],
        "downstream": ["05"],
    },
    "05": {
        "notebook": CANONICAL_NOTEBOOKS[4],
        "output_root_hint": "03_results/05_external_validation_and_H3_vFINAL",
        "control_artifacts": [
            "ENVIRONMENT_PREFLIGHT.json", "INFERENCE_CACHE_PROVENANCE.json", "CACHE_SCHEMA_VALIDATION.json",
            "LOCKED_SOURCE_TEST_PREDICTIONS.parquet", "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
            "BAITBUSTER_HUMAN_PREDICTIONS.parquet", "H1_H2_PARITY_REPORT.json",
            "H1_RESULTS.json", "H2_RESULTS.json", "SENSITIVITY_RESULTS.json", "H3_RESULTS.json",
            "H3_SPARSITY_REPORT.csv", "H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv",
            "H3_BAITBUSTER_REPLICATION_CORRECTED.csv", "H3_PERTURBATION_PAIRS.parquet",
            "H3_PERTURBATION_EXCLUSIONS.csv", "H3P_POINT_PARITY.json", "H3P_STATISTICAL_PARITY.json",
            "H3_PERTURBATION_RESULTS.json", "STATISTICAL_SUMMARY.json", "RUN_METADATA.json",
            "STAGE05_SUMMARY.md", "STAGE05_ARTIFACT_MANIFEST.json", "STAGE05_SHA256SUMS.txt",
            "STAGE_STATUS.json",
        ],
        "required_data_interfaces": [
            "LOCKED_SOURCE_TEST_PREDICTIONS.parquet",
            "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
            "BAITBUSTER_HUMAN_PREDICTIONS.parquet",
        ],
        "downstream": ["06", "07", "08", "09"],
    },
    "06": {
        "notebook": CANONICAL_NOTEBOOKS[5],
        "output_root_hint": "03_results/06_fixed_budget_reliability_evaluation",
        "control_artifacts": [
            "ENVIRONMENT_PREFLIGHT.json", "UPSTREAM_PROVENANCE_GATE.json", "H4_RESULTS.json",
            "FIXED_BUDGET_TABLES.csv", "RISK_COVERAGE_DATA.csv", "BOOTSTRAP_RUNTIME.json",
            "FINAL_STATISTICAL_SUMMARY.json", "RUN_METADATA.json", "STAGE06_SUMMARY.md",
            "REPRODUCIBILITY_MANIFEST.json", "STAGE06_SHA256SUMS.txt", "STAGE_STATUS.json",
        ],
        "required_data_interfaces": [],
        "downstream": ["07", "08", "09"],
    },
    "07": {
        "notebook": CANONICAL_NOTEBOOKS[6],
        "output_root_hint": "03_results/07_H3_H4_BRIDGE_EXPLORATORY_vFINAL",
        "control_artifacts": [
            "UPSTREAM_PROVENANCE_GATE.json", "FROZEN_INPUT_SUMMARY.json", "H3_POINT_PARITY.json",
            "H4_POLICY_POINT_PARITY.json", "BRIDGE_32_CELL_POINT_TABLE.csv",
            "BRIDGE_POINT_ASSOCIATIONS.json", "H3_H4_BRIDGE_RESULTS.json", "BOOTSTRAP_RUNTIME.json",
            "LEAVE_ONE_PLATFORM_OUT_POINT_TABLE.csv", "LEAVE_ONE_PLATFORM_OUT_ROBUSTNESS.json",
            "STAGE07_SUMMARY.md", "REPRODUCIBILITY_MANIFEST.json", "STAGE07_SHA256SUMS.txt",
            "STAGE_STATUS.json",
        ],
        "required_data_interfaces": [],
        "downstream": ["08", "09"],
    },
    "08": {
        "notebook": CANONICAL_NOTEBOOKS[7],
        "output_root_hint": "08_manuscript_outputs",
        "control_artifacts": [
            "PROVENANCE_REPORT.json", "INPUT_HASHES.json", "SCHEMA_VALIDATION.json",
            "FIGURE_SOURCE_COVERAGE.json", "MANIFEST.json", "STAGE08_SHA256SUMS.txt",
            "STAGE08_STATUS.json",
        ],
        "required_data_interfaces": [],
        "downstream": ["09", "manuscript"],
    },
    "09": {
        "notebook": CANONICAL_NOTEBOOKS[8],
        "output_root_hint": "09_results/09_SOURCE_PUBLISHER_CLUSTER_SENSITIVITY_vFINAL",
        "control_artifacts": [
            "SOURCE_PUBLISHER_MAPPING.csv", "SOURCE_CLUSTER_DIAGNOSTICS.json",
            "H1_H2_SOURCE_CLUSTER_SENSITIVITY.json", "H3_SOURCE_CLUSTER_SENSITIVITY.json",
            "H3P_SOURCE_CLUSTER_SENSITIVITY.json", "H4_SOURCE_CLUSTER_SENSITIVITY.json",
            "CLAIM_CONCORDANCE_TABLE.csv", "SENSITIVITY_SUMMARY.md",
            "REPRODUCIBILITY_MANIFEST.json", "STAGE09_SHA256SUMS.txt", "STAGE_STATUS.json",
        ],
        "required_data_interfaces": [],
        "downstream": ["manuscript_sensitivity_only"],
    },
}

# Assert that every explicit contract artifact was actually declared by its canonical notebook source.
contract_scan_failures = []
for stage, spec in ARTIFACT_CONTRACTS.items():
    declared = set(NOTEBOOK_STATIC_SCAN[stage]["file_literals"])
    for rel in spec["control_artifacts"] + spec["required_data_interfaces"]:
        basename = Path(rel).name
        if rel not in declared and basename not in {Path(x).name for x in declared}:
            # Some nested clean-split names are generated by a format string; keep them only when
            # their governing manifest is directly declared.
            if not (stage == "03" and rel.startswith("clean_splits/") and "CLEAN_DATA_MANIFEST.json" in declared):
                contract_scan_failures.append({"stage": stage, "artifact": rel})
if contract_scan_failures:
    hard_stop("Explicit release contract contains artifacts not supported by notebook static scan",
              {"failures": contract_scan_failures})

dependency_rows = []
role_overrides = {
    "LOCKED_SOURCE_TEST_PREDICTIONS.parquet": "locked downstream analysis interface: source TEST predictions",
    "BANGLACLICK_PLATFORM_PREDICTIONS.parquet": "locked downstream analysis interface: BanglaClick predictions",
    "BAITBUSTER_HUMAN_PREDICTIONS.parquet": "locked downstream analysis interface: BaitBuster human predictions",
    "MODEL_ARTIFACT_MANIFEST.json": "Stage-03 model artifact inventory and hashes",
    "CLEAN_DATA_MANIFEST.json": "Stage-03 decontaminated split inventory and hashes",
    "MODEL_CONFIG_LOCK.json": "locked model/configuration specification",
    "STAGE09_SHA256SUMS.txt": "Stage-09 sensitivity checksum contract",
}
for stage, spec in ARTIFACT_CONTRACTS.items():
    for rel in spec["control_artifacts"] + spec["required_data_interfaces"]:
        dependency_rows.append({
            "stage": stage,
            "notebook": spec["notebook"],
            "output_root": spec["output_root_hint"],
            "required_artifact": rel,
            "artifact_role": role_overrides.get(rel, "canonical stage release artifact"),
            "downstream_consumers": spec["downstream"],
            "required_for_scientific_reproduction": True,
            "optional_large_artifact": False,
            "canonical_status": "CANONICAL" if stage != "09" else "CANONICAL_SENSITIVITY_ONLY",
        })
write_json(RELEASE_ROOT / "STAGE_ARTIFACT_DEPENDENCY_MAP.json", dependency_rows)
print("Static artifact/dependency contract: PASS")


Static artifact/dependency contract: PASS


## 5. Discover and cryptographically verify existing Save-Version outputs

Discovery is content-gated. No scientific stage is invoked. Stages 03/04 are bound through the canonical Stage 05 upstream identities; Stage 09 is admitted only as the unique valid final sensitivity identity.


In [6]:

def candidate_parent_dirs(filename):
    parents = []
    for p in find_files_named(filename):
        parents.append(p.parent.resolve())
    return sorted(set(parents), key=str)

def discover_pinned_stage(expected, required_names, label):
    matches = []
    for p in find_files_named(expected["manifest_name"]):
        try:
            if sha256_file(p) != expected["manifest_sha256"]:
                continue
            root = p.parent.resolve()
            if not all((root / x).is_file() for x in required_names):
                continue
            if sha256_file(root / expected["checksum_name"]) != expected["checksum_sha256"]:
                continue
            if sha256_file(root / expected["status_name"]) != expected["status_sha256"]:
                continue
            matches.append(root)
        except Exception:
            continue
    if not matches:
        hard_stop(f"{label} exact canonical root not found", {"expected": expected})
    chosen = sorted(set(matches), key=str)[0]
    verify_checksum_contract(chosen, expected["checksum_name"], require_all=True)
    return chosen, sorted(set(matches), key=str)

# First bind canonical Stage 05 because it carries the exact Stage 03/04 checksum identities.
S5_ROOT, S5_ALIASES = discover_pinned_stage(
    EXPECTED_STAGE05,
    ARTIFACT_CONTRACTS["05"]["control_artifacts"],
    "Stage-05",
)
s5_manifest, _ = verify_manifest_payload_seal(S5_ROOT / "STAGE05_ARTIFACT_MANIFEST.json")
if not (
    s5_manifest.get("stage_id") == "05"
    and s5_manifest.get("canonical_stage05") is True
    and s5_manifest.get("canonical_version") == EXPECTED_STAGE05["canonical_version"]
    and s5_manifest.get("protocol_version") == PROTOCOL_VERSION
):
    hard_stop("Stage-05 semantic canonical identity mismatch")

for rel, expected_hash in EXPECTED_STAGE05["prediction_sha256"].items():
    if sha256_file(S5_ROOT / rel) != expected_hash:
        hard_stop("Stage-05 locked prediction identity mismatch", {"artifact": rel})

S3_CHECKSUM_EXPECTED = s5_manifest.get("upstream_stage03_checksum_manifest_sha256")
S4_CHECKSUM_EXPECTED = s5_manifest.get("upstream_stage04_checksum_manifest_sha256")
if not re.fullmatch(r"[0-9a-f]{64}", str(S3_CHECKSUM_EXPECTED or "")):
    hard_stop("Stage-05 manifest lacks valid Stage-03 checksum identity")
if not re.fullmatch(r"[0-9a-f]{64}", str(S4_CHECKSUM_EXPECTED or "")):
    hard_stop("Stage-05 manifest lacks valid Stage-04 checksum identity")

s3_sum_path, s3_sum_aliases = locate_file_by_hash("STAGE03_SHA256SUMS.txt", S3_CHECKSUM_EXPECTED)
S3_ROOT = s3_sum_path.parent.resolve()
verify_checksum_contract(S3_ROOT, "STAGE03_SHA256SUMS.txt", require_all=True)
if json.loads((S3_ROOT / "STAGE_STATUS.json").read_text(encoding="utf-8")).get("status") != "PASS":
    hard_stop("Stage-03 status is not PASS")

s4_sum_path, s4_sum_aliases = locate_file_by_hash("STAGE04_SHA256SUMS.txt", S4_CHECKSUM_EXPECTED)
S4_ROOT = s4_sum_path.parent.resolve()
verify_checksum_contract(S4_ROOT, "STAGE04_SHA256SUMS.txt", require_all=True)
if json.loads((S4_ROOT / "STAGE_STATUS.json").read_text(encoding="utf-8")).get("status") != "PASS":
    hard_stop("Stage-04 status is not PASS")

S6_ROOT, S6_ALIASES = discover_pinned_stage(
    EXPECTED_STAGE06, ARTIFACT_CONTRACTS["06"]["control_artifacts"], "Stage-06"
)
if sha256_file(S6_ROOT / "H4_RESULTS.json") != EXPECTED_STAGE06["h4_sha256"]:
    hard_stop("Stage-06 H4 identity mismatch")

S7_ROOT, S7_ALIASES = discover_pinned_stage(
    EXPECTED_STAGE07, ARTIFACT_CONTRACTS["07"]["control_artifacts"], "Stage-07"
)
S8_ROOT, S8_ALIASES = discover_pinned_stage(
    EXPECTED_STAGE08, ARTIFACT_CONTRACTS["08"]["control_artifacts"], "Stage-08 FINAL2 publication execution"
)

# Stage 09 has no downstream stage to provide an external file hash pin.
# Require a unique payload-sealed, checksum-valid, PASS sensitivity identity.
stage09_candidates = []
for p in find_files_named("REPRODUCIBILITY_MANIFEST.json"):
    try:
        man, seal = verify_manifest_payload_seal(p)
        if man.get("stage_id") != "09":
            continue
        if man.get("protocol_version") != PROTOCOL_VERSION:
            continue
        if man.get("sensitivity_version") != EXPECTED_STAGE09_SENSITIVITY_VERSION:
            continue
        if man.get("scientific_role") != EXPECTED_STAGE09_SOURCE_ROLE:
            continue
        root = p.parent.resolve()
        if not all((root / x).is_file() for x in ARTIFACT_CONTRACTS["09"]["control_artifacts"]):
            continue
        status = json.loads((root / "STAGE_STATUS.json").read_text(encoding="utf-8"))
        if status.get("status") != "PASS":
            continue
        verify_checksum_contract(root, "STAGE09_SHA256SUMS.txt", require_all=True)
        stage09_candidates.append({
            "root": root,
            "manifest_sha256": sha256_file(root / "REPRODUCIBILITY_MANIFEST.json"),
            "checksum_sha256": sha256_file(root / "STAGE09_SHA256SUMS.txt"),
            "status_sha256": sha256_file(root / "STAGE_STATUS.json"),
        })
    except Exception:
        continue

if not stage09_candidates:
    hard_stop("No valid final Stage-09 sensitivity root found")
stage09_identity_groups = defaultdict(list)
for rec in stage09_candidates:
    key = (rec["manifest_sha256"], rec["checksum_sha256"], rec["status_sha256"])
    stage09_identity_groups[key].append(rec["root"])
if len(stage09_identity_groups) != 1:
    hard_stop("Multiple distinct valid Stage-09 identities found; refusing to guess",
              {"identities": {str(k): [str(x) for x in v] for k, v in stage09_identity_groups.items()}})
S9_IDENTITY = next(iter(stage09_identity_groups))
S9_ROOT = sorted(stage09_identity_groups[S9_IDENTITY], key=str)[0]
S9_ALIASES = sorted(stage09_identity_groups[S9_IDENTITY], key=str)

# Stage 01: authenticate either the complete extracted freeze or the exact frozen ZIP.
# This reproduces the Stage-03 frozen content-identity contract exactly. In particular,
# README_HANDOFF.txt is part of the five-control identity payload used by Stage 03.
STAGE01_REQUIRED_CONTROLS = [
    "ACQUISITION_MANIFEST.json",
    "ACQUISITION_STATUS.json",
    "SOURCES_LOCK.json",
    "SHA256SUMS.txt",
    "README_HANDOFF.txt",
]
STAGE01_REQUIRED_DATASET_DIRS = [
    "BaitBuster-Bangla", "BanFakeNews-2.0", "BanMANI", "BanglaBait", "BanglaClick"
]
STAGE01_ZIP_NAME = "EXTERNAL-DATASET-BYTES-v1.zip"


def inspect_frozen_root(root):
    """Validate one extracted Stage-01 root using the exact Stage-03 identity payload."""
    root = Path(root).resolve()
    record = {
        "root": str(root),
        "complete_required_tree": False,
        "checksum_valid": False,
        "actual_content_identity": None,
        "reason": None,
    }
    missing_controls = [name for name in STAGE01_REQUIRED_CONTROLS if not (root / name).is_file()]
    missing_dirs = [name for name in STAGE01_REQUIRED_DATASET_DIRS if not (root / name).is_dir()]
    if missing_controls or missing_dirs:
        record["reason"] = {
            "type": "INCOMPLETE_TREE",
            "missing_controls": missing_controls,
            "missing_dataset_dirs": missing_dirs,
        }
        return record
    record["complete_required_tree"] = True
    try:
        sums, failures = verify_checksum_contract(root, "SHA256SUMS.txt", require_all=False)
    except Exception as exc:
        record["reason"] = {"type": "CHECKSUM_CONTRACT_ERROR", "detail": f"{type(exc).__name__}: {exc}"}
        return record
    if failures:
        record["reason"] = {"type": "CHECKSUM_FAILURE", "failures": failures[:20]}
        return record
    record["checksum_valid"] = True
    controls = {name: sha256_file(root / name) for name in STAGE01_REQUIRED_CONTROLS}
    payload = {
        "schema_version": 1,
        "freeze_id": "EXTERNAL-DATASET-BYTES-v1",
        "controls": controls,
        "declared_sha256sums": sorted(sums.items()),
        "dataset_dirs": sorted(STAGE01_REQUIRED_DATASET_DIRS),
    }
    record["actual_content_identity"] = hashlib.sha256(canonical_json_bytes(payload)).hexdigest()
    return record


def safe_extract_stage01_zip(zip_path, destination):
    """Extract only an already SHA-pinned Stage-01 ZIP, with CRC and traversal checks."""
    zip_path = Path(zip_path).resolve()
    destination = Path(destination).resolve()
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        bad_member = zf.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Stage-01 ZIP CRC verification failed at member: {bad_member}")
        for info in zf.infolist():
            target = (destination / info.filename).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe Stage-01 ZIP member path: {info.filename}")
        zf.extractall(destination)


def print_stage01_input_diagnostic(diag):
    print("STAGE01_INPUT_FORM:", diag.get("STAGE01_INPUT_FORM"))
    print("ZIP_CANDIDATES:", json.dumps(diag.get("ZIP_CANDIDATES", []), ensure_ascii=False, sort_keys=True))
    print("EXTRACTED_ROOT_CANDIDATES:", json.dumps(diag.get("EXTRACTED_ROOT_CANDIDATES", []), ensure_ascii=False, sort_keys=True))
    print("EXPECTED_ZIP_SHA256:", diag.get("EXPECTED_ZIP_SHA256"))
    print("ACTUAL_ZIP_SHA256:", diag.get("ACTUAL_ZIP_SHA256"))
    print("EXPECTED_CONTENT_IDENTITY:", diag.get("EXPECTED_CONTENT_IDENTITY"))
    print("ACTUAL_CONTENT_IDENTITY:", diag.get("ACTUAL_CONTENT_IDENTITY"))
    print("SELECTED_STAGE01_ROOT:", diag.get("SELECTED_STAGE01_ROOT"))


s3_input_identity = json.loads((S3_ROOT / "INPUT_IDENTITY.json").read_text(encoding="utf-8"))
expected_frozen_content_identity = s3_input_identity.get("content_identity_sha256")
if not re.fullmatch(r"[0-9a-f]{64}", str(expected_frozen_content_identity or "")):
    hard_stop("Stage-03 INPUT_IDENTITY.json lacks a valid frozen content identity")

# Inspect every directly mounted extracted-root candidate. Filename locates candidates only;
# acceptance requires the complete tree, checksum verification, and exact frozen content identity.
extracted_candidate_records = []
seen_extracted_roots = set()
for p in find_files_named("ACQUISITION_MANIFEST.json"):
    root = p.parent.resolve()
    if root in seen_extracted_roots:
        continue
    seen_extracted_roots.add(root)
    rec = inspect_frozen_root(root)
    rec["candidate_source"] = str(p)
    rec["input_form"] = "EXTRACTED_DIRECTORY"
    extracted_candidate_records.append(rec)

# Inspect every literal frozen-ZIP candidate. No ZIP is extracted unless its bytes exactly match
# the independently frozen Stage-01 ZIP SHA-256.
zip_candidate_records = []
verified_zip_records = []
stage01_temp_base = work_root / ".stage10_stage01_input"
for idx, zp in enumerate(find_files_named(STAGE01_ZIP_NAME)):
    zrec = {
        "path": str(zp),
        "actual_zip_sha256": None,
        "expected_zip_sha256": EXPECTED_FROZEN_ZIP_SHA256,
        "zip_hash_match": False,
        "extracted_root": None,
        "complete_required_tree": False,
        "actual_content_identity": None,
        "content_identity_match": False,
        "reason": None,
    }
    try:
        zhash = sha256_file(zp)
        zrec["actual_zip_sha256"] = zhash
        zrec["zip_hash_match"] = zhash == EXPECTED_FROZEN_ZIP_SHA256
        if not zrec["zip_hash_match"]:
            zrec["reason"] = "ZIP_SHA256_MISMATCH"
            zip_candidate_records.append(zrec)
            continue

        extract_dir = stage01_temp_base / f"zip_{idx:03d}_{zhash[:16]}"
        safe_extract_stage01_zip(zp, extract_dir)
        structural_roots = sorted(
            {m.parent.resolve() for m in extract_dir.rglob("ACQUISITION_MANIFEST.json") if m.is_file()},
            key=str,
        )
        root_records = [inspect_frozen_root(root) for root in structural_roots]
        complete_records = [r for r in root_records if r["complete_required_tree"] and r["checksum_valid"]]
        if len(complete_records) != 1:
            zrec["reason"] = {
                "type": "ZIP_EXTRACTED_ROOT_CARDINALITY",
                "complete_checksum_valid_roots": complete_records,
                "all_structural_roots": root_records,
            }
            zip_candidate_records.append(zrec)
            continue

        root_rec = complete_records[0]
        zrec["extracted_root"] = root_rec["root"]
        zrec["complete_required_tree"] = True
        zrec["actual_content_identity"] = root_rec["actual_content_identity"]
        zrec["content_identity_match"] = root_rec["actual_content_identity"] == expected_frozen_content_identity
        if not zrec["content_identity_match"]:
            zrec["reason"] = "ZIP_CONTENT_IDENTITY_MISMATCH"
            zip_candidate_records.append(zrec)
            continue
        verified_zip_records.append(dict(zrec))
    except Exception as exc:
        zrec["reason"] = f"{type(exc).__name__}: {exc}"
    zip_candidate_records.append(zrec)

# Structurally complete, checksum-valid extracted trees are independently authenticated by the
# same frozen content identity. Keep all candidates for provenance and detect conflicts explicitly.
complete_extracted_records = [
    r for r in extracted_candidate_records if r["complete_required_tree"] and r["checksum_valid"]
]
valid_extracted_records = [
    r for r in complete_extracted_records if r["actual_content_identity"] == expected_frozen_content_identity
]
conflicting_extracted_records = [
    r for r in complete_extracted_records if r["actual_content_identity"] != expected_frozen_content_identity
]

# An exact frozen ZIP whose extracted bytes do not reproduce the Stage-03 identity is a fatal
# integrity contradiction, not a fallback opportunity.
exact_zip_identity_conflicts = [
    r for r in zip_candidate_records
    if r.get("zip_hash_match") and r.get("actual_content_identity") is not None and not r.get("content_identity_match")
]
if exact_zip_identity_conflicts:
    diag = {
        "STAGE01_INPUT_FORM": None,
        "ZIP_CANDIDATES": zip_candidate_records,
        "EXTRACTED_ROOT_CANDIDATES": extracted_candidate_records,
        "EXPECTED_ZIP_SHA256": EXPECTED_FROZEN_ZIP_SHA256,
        "ACTUAL_ZIP_SHA256": [r.get("actual_zip_sha256") for r in zip_candidate_records],
        "EXPECTED_CONTENT_IDENTITY": expected_frozen_content_identity,
        "ACTUAL_CONTENT_IDENTITY": [
            r.get("actual_content_identity") for r in extracted_candidate_records + zip_candidate_records
            if r.get("actual_content_identity") is not None
        ],
        "SELECTED_STAGE01_ROOT": None,
        "reason": "EXACT_FROZEN_ZIP_CONTENT_IDENTITY_CONTRADICTION",
    }
    write_json(RELEASE_ROOT / "STAGE01_INPUT_FORM_DIAGNOSTIC.json", diag)
    print_stage01_input_diagnostic(diag)
    hard_stop("Exact frozen Stage-01 ZIP does not reproduce the frozen Stage-03 content identity", diag)

# If a valid expected identity and a different structurally valid extracted identity coexist,
# do not guess which package is intended.
if (valid_extracted_records or verified_zip_records) and conflicting_extracted_records:
    diag = {
        "STAGE01_INPUT_FORM": None,
        "ZIP_CANDIDATES": zip_candidate_records,
        "EXTRACTED_ROOT_CANDIDATES": extracted_candidate_records,
        "EXPECTED_ZIP_SHA256": EXPECTED_FROZEN_ZIP_SHA256,
        "ACTUAL_ZIP_SHA256": [r.get("actual_zip_sha256") for r in zip_candidate_records],
        "EXPECTED_CONTENT_IDENTITY": expected_frozen_content_identity,
        "ACTUAL_CONTENT_IDENTITY": sorted({
            str(r.get("actual_content_identity")) for r in complete_extracted_records + zip_candidate_records
            if r.get("actual_content_identity") is not None
        }),
        "SELECTED_STAGE01_ROOT": None,
        "reason": "CONFLICTING_COMPLETE_STAGE01_IDENTITIES",
    }
    write_json(RELEASE_ROOT / "STAGE01_INPUT_FORM_DIAGNOSTIC.json", diag)
    print_stage01_input_diagnostic(diag)
    hard_stop("Conflicting complete Stage-01 frozen identities found", diag)

# Deterministic preference: if both authenticated forms represent the same frozen identity,
# prefer the already-extracted mounted root to avoid unnecessary dependence on temporary extraction.
if valid_extracted_records:
    selected = sorted(valid_extracted_records, key=lambda r: r["root"])[0]
    S1_ROOT = Path(selected["root"]).resolve()
    STAGE01_INPUT_FORM = "EXTRACTED_DIRECTORY"
    selected_zip_sha = None
elif verified_zip_records:
    selected = sorted(verified_zip_records, key=lambda r: r["path"])[0]
    S1_ROOT = Path(selected["extracted_root"]).resolve()
    STAGE01_INPUT_FORM = "ZIP"
    selected_zip_sha = selected["actual_zip_sha256"]
else:
    diag = {
        "STAGE01_INPUT_FORM": None,
        "ZIP_CANDIDATES": zip_candidate_records,
        "EXTRACTED_ROOT_CANDIDATES": extracted_candidate_records,
        "EXPECTED_ZIP_SHA256": EXPECTED_FROZEN_ZIP_SHA256,
        "ACTUAL_ZIP_SHA256": [r.get("actual_zip_sha256") for r in zip_candidate_records],
        "EXPECTED_CONTENT_IDENTITY": expected_frozen_content_identity,
        "ACTUAL_CONTENT_IDENTITY": [
            r.get("actual_content_identity") for r in extracted_candidate_records + zip_candidate_records
            if r.get("actual_content_identity") is not None
        ],
        "SELECTED_STAGE01_ROOT": None,
        "reason": "NO_AUTHENTICATED_STAGE01_INPUT_FORM",
    }
    write_json(RELEASE_ROOT / "STAGE01_INPUT_FORM_DIAGNOSTIC.json", diag)
    print_stage01_input_diagnostic(diag)
    hard_stop("Exact frozen Stage-01 input not found in extracted-directory or pinned-ZIP form", diag)

# Retain both forms/aliases in provenance when they authenticate to the same frozen identity.
S1_ALIASES = sorted({Path(r["root"]).resolve() for r in valid_extracted_records}, key=str)
S1_ZIP_ALIASES = sorted({Path(r["path"]).resolve() for r in verified_zip_records}, key=str)
selected_identity = inspect_frozen_root(S1_ROOT)["actual_content_identity"]
STAGE01_INPUT_AUDIT = {
    "input_form": STAGE01_INPUT_FORM,
    "selected_stage01_root": str(S1_ROOT),
    "selected_zip_sha256": selected_zip_sha,
    "expected_zip_sha256": EXPECTED_FROZEN_ZIP_SHA256,
    "expected_content_identity": expected_frozen_content_identity,
    "actual_content_identity": selected_identity,
    "extracted_root_aliases": [str(x) for x in S1_ALIASES],
    "verified_zip_aliases": [str(x) for x in S1_ZIP_ALIASES],
    "zip_candidates": zip_candidate_records,
    "extracted_root_candidates": extracted_candidate_records,
    "selection_rule": (
        "Authenticate extracted roots by complete tree + checksum contract + exact Stage-03 content identity; "
        "authenticate ZIPs by exact frozen ZIP SHA-256, safe extraction, complete tree, checksum contract, and exact "
        "Stage-03 content identity. When both forms authenticate to the same identity, prefer the already-extracted root."
    ),
}
write_json(RELEASE_ROOT / "STAGE01_INPUT_FORM_DIAGNOSTIC.json", {
    "STAGE01_INPUT_FORM": STAGE01_INPUT_FORM,
    "ZIP_CANDIDATES": zip_candidate_records,
    "EXTRACTED_ROOT_CANDIDATES": extracted_candidate_records,
    "EXPECTED_ZIP_SHA256": EXPECTED_FROZEN_ZIP_SHA256,
    "ACTUAL_ZIP_SHA256": [r.get("actual_zip_sha256") for r in zip_candidate_records],
    "EXPECTED_CONTENT_IDENTITY": expected_frozen_content_identity,
    "ACTUAL_CONTENT_IDENTITY": selected_identity,
    "SELECTED_STAGE01_ROOT": str(S1_ROOT),
    "VERIFIED_ZIP_ALIASES": [str(x) for x in S1_ZIP_ALIASES],
    "EXTRACTED_ROOT_ALIASES": [str(x) for x in S1_ALIASES],
    "SELECTION_RULE": STAGE01_INPUT_AUDIT["selection_rule"],
})
print_stage01_input_diagnostic({
    "STAGE01_INPUT_FORM": STAGE01_INPUT_FORM,
    "ZIP_CANDIDATES": zip_candidate_records,
    "EXTRACTED_ROOT_CANDIDATES": extracted_candidate_records,
    "EXPECTED_ZIP_SHA256": EXPECTED_FROZEN_ZIP_SHA256,
    "ACTUAL_ZIP_SHA256": [r.get("actual_zip_sha256") for r in zip_candidate_records],
    "EXPECTED_CONTENT_IDENTITY": expected_frozen_content_identity,
    "ACTUAL_CONTENT_IDENTITY": selected_identity,
    "SELECTED_STAGE01_ROOT": str(S1_ROOT),
})

# Existing downstream Stage-01 checks remain unchanged after authenticated root selection.
s1_status = json.loads((S1_ROOT / "ACQUISITION_STATUS.json").read_text(encoding="utf-8"))
if s1_status.get("package", {}).get("status") != "PASS":
    hard_stop("Stage-01 frozen package status is not PASS")
test_path = S1_ROOT / "BanglaBait" / "repository_snapshot" / "test.csv"
if not test_path.is_file() or sha256_file(test_path) != EXPECTED_BANGLABAIT_TEST_SHA256:
    hard_stop("Stage-01 frozen BanglaBait TEST identity mismatch")

# Stage 02: bind the exact methodology identity consumed by Stage 03.
stage02_roots = []
for p in find_files_named("P0_RUN_METADATA.json"):
    try:
        meta = json.loads(p.read_text(encoding="utf-8"))
        if meta.get("stage_id") != "02":
            continue
        if meta.get("methodology_sha256") != EXPECTED_STAGE02_METHODOLOGY_SHA256:
            continue
        root = p.parent.resolve()
        if not all((root / x).is_file() for x in ARTIFACT_CONTRACTS["02"]["control_artifacts"]):
            continue
        verify_checksum_contract(root, "P0_OUTPUT_SHA256SUMS.txt", require_all=True)
        stage02_roots.append(root)
    except Exception:
        continue
if not stage02_roots:
    hard_stop("Canonical Stage-02 output root not found by methodology identity")
# Distinct Stage-02 runs with different output-checksum identities are ambiguous.
s2_groups = defaultdict(list)
for root in stage02_roots:
    s2_groups[sha256_file(root / "P0_OUTPUT_SHA256SUMS.txt")].append(root)
if len(s2_groups) != 1:
    hard_stop("Multiple distinct Stage-02 output identities found; refusing to guess",
              {"checksum_identities": {k: [str(x) for x in v] for k, v in s2_groups.items()}})
S2_ROOT = sorted(next(iter(s2_groups.values())), key=str)[0]
S2_ALIASES = sorted(next(iter(s2_groups.values())), key=str)

STAGE_ROOTS = {
    "01": S1_ROOT, "02": S2_ROOT, "03": S3_ROOT, "04": S4_ROOT,
    "05": S5_ROOT, "06": S6_ROOT, "07": S7_ROOT, "08": S8_ROOT, "09": S9_ROOT,
}
STAGE_ALIASES = {
    "01": S1_ALIASES, "02": S2_ALIASES, "03": [p.parent for p in s3_sum_aliases],
    "04": [p.parent for p in s4_sum_aliases], "05": S5_ALIASES, "06": S6_ALIASES,
    "07": S7_ALIASES, "08": S8_ALIASES, "09": S9_ALIASES,
}
write_json(RELEASE_ROOT / "DISCOVERED_STAGE_ROOTS.json", {
    stage: {"selected": str(root), "equivalent_candidates": [str(x) for x in STAGE_ALIASES[stage]]}
    for stage, root in STAGE_ROOTS.items()
})
print("Existing Save-Version discovery + integrity gates: PASS (Stages 01–09)")


STAGE01_INPUT_FORM: EXTRACTED_DIRECTORY
ZIP_CANDIDATES: [{"actual_content_identity": "161b6e091b3db55a9616c943693ad456a175c19330b4297b5e068910b47c2b8f", "actual_zip_sha256": "e0de5c224f406fe22d8ad30804217465f789c49c70e1d3afbe00afd28deb7ccf", "complete_required_tree": true, "content_identity_match": true, "expected_zip_sha256": "e0de5c224f406fe22d8ad30804217465f789c49c70e1d3afbe00afd28deb7ccf", "extracted_root": "/kaggle/working/.stage10_stage01_input/zip_000_e0de5c224f406fe2/EXTERNAL-DATASET-BYTES-v1", "path": "/kaggle/input/notebooks/smalakarishere/01-dataset-acquisition-and-freeze/EXTERNAL-DATASET-BYTES-v1.zip", "reason": null, "zip_hash_match": true}]
EXTRACTED_ROOT_CANDIDATES: [{"actual_content_identity": "161b6e091b3db55a9616c943693ad456a175c19330b4297b5e068910b47c2b8f", "candidate_source": "/kaggle/input/notebooks/smalakarishere/01-dataset-acquisition-and-freeze/EXTERNAL-DATASET-BYTES-v1/ACQUISITION_MANIFEST.json", "checksum_valid": true, "complete_required_tree": true, "input_fo

## 6. Stage 03 core vs optional large training artifacts

In [7]:

s3_model_manifest, _ = verify_manifest_payload_seal(S3_ROOT / "MODEL_ARTIFACT_MANIFEST.json")
s3_clean_manifest = json.loads((S3_ROOT / "CLEAN_DATA_MANIFEST.json").read_text(encoding="utf-8"))
s3_sums = parse_sha256sums(S3_ROOT / "STAGE03_SHA256SUMS.txt")

OPTIONAL_MODEL_KEYS = {"banglabert", "xlmr"}
optional_large_rows = []
stage3_optional_paths = set()
stage3_small_model_paths = set()

for rec in s3_model_manifest.get("models", []):
    model = str(rec.get("model"))
    artifact_path = str(rec.get("artifact_path", "")).replace("\\", "/")
    files = rec.get("files", [])
    target_set = stage3_optional_paths if model in OPTIONAL_MODEL_KEYS else stage3_small_model_paths
    for f in files:
        rel = str(f.get("path", "")).replace("\\", "/")
        if rel:
            target_set.add(rel)
            if model in OPTIONAL_MODEL_KEYS:
                src = S3_ROOT / rel
                optional_large_rows.append({
                    "path": rel,
                    "size_bytes": int(f.get("size", src.stat().st_size if src.exists() else 0)),
                    "sha256_if_available": f.get("sha256") or (sha256_file(src) if src.is_file() else ""),
                    "artifact_type": f"{model}_checkpoint_or_tokenizer_artifact",
                    "required_for_reported_numbers": False,
                    "required_for_full_training_reproduction": True,
                    "license_note": "REQUIRES AUTHOR VERIFICATION",
                    "release_action": (
                        "COPY_TO_models_optional" if COPY_OPTIONAL_LARGE_TRAINING_ARTIFACTS
                        else "DOCUMENT_ONLY_OR_EXTERNALLY_PIN"
                    ),
                })

# Include any large model/training-state file omitted from manifest classification.
for rel, digest in s3_sums.items():
    if not rel.startswith("models/"):
        continue
    p = S3_ROOT / rel
    if rel in stage3_optional_paths or rel in stage3_small_model_paths:
        continue
    if p.is_file() and p.stat().st_size >= 100 * 1024 * 1024:
        stage3_optional_paths.add(rel)
        optional_large_rows.append({
            "path": rel,
            "size_bytes": p.stat().st_size,
            "sha256_if_available": digest,
            "artifact_type": "other_large_training_state",
            "required_for_reported_numbers": False,
            "required_for_full_training_reproduction": True,
            "license_note": "REQUIRES AUTHOR VERIFICATION",
            "release_action": (
                "COPY_TO_models_optional" if COPY_OPTIONAL_LARGE_TRAINING_ARTIFACTS
                else "DOCUMENT_ONLY_OR_EXTERNALLY_PIN"
            ),
        })

with (RELEASE_ROOT / "OPTIONAL_LARGE_TRAINING_ARTIFACTS.csv").open("w", encoding="utf-8", newline="") as f:
    fields = [
        "path", "size_bytes", "sha256_if_available", "artifact_type",
        "required_for_reported_numbers", "required_for_full_training_reproduction",
        "license_note", "release_action",
    ]
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for row in sorted(optional_large_rows, key=lambda r: r["path"]):
        w.writerow(row)

print("Stage-03 optional large artifacts documented:", len(optional_large_rows))


# Add optional Stage-03 training artifacts to the dependency map explicitly.
dependency_map_path = RELEASE_ROOT / "STAGE_ARTIFACT_DEPENDENCY_MAP.json"
dependency_map = json.loads(dependency_map_path.read_text(encoding="utf-8"))
for row in optional_large_rows:
    dependency_map.append({
        "stage": "03",
        "notebook": ARTIFACT_CONTRACTS["03"]["notebook"],
        "output_root": ARTIFACT_CONTRACTS["03"]["output_root_hint"],
        "required_artifact": row["path"],
        "artifact_role": row["artifact_type"],
        "downstream_consumers": ["full_training_reproduction_only"],
        "required_for_scientific_reproduction": False,
        "optional_large_artifact": True,
        "canonical_status": "CANONICAL_OPTIONAL_LARGE_TRAINING_ARTIFACT",
    })
write_json(dependency_map_path, dependency_map)


Stage-03 optional large artifacts documented: 40


## 7. Copy verified canonical artifacts

The core release copies all checksum-contracted Stage 03–09 outputs except explicitly optional large transformer training artifacts. Stage 01 raw dataset bytes are **not redistributed by default** until licenses are author-verified; Stage 01 control metadata and exact hashes are included. Stage 02 outputs are copied from its verified output checksum contract.


In [8]:

COPIED_RECORDS = []
NOT_COPIED_RECORDS = []

def record_copy(stage, src, dst, expected, role):
    observed = safe_copy_verified(src, dst, expected)
    COPIED_RECORDS.append({
        "stage": stage,
        "source_relative": str(Path(src).relative_to(STAGE_ROOTS[stage])).replace("\\", "/"),
        "release_relative": str(Path(dst).relative_to(RELEASE_ROOT)).replace("\\", "/"),
        "sha256": observed,
        "size_bytes": Path(dst).stat().st_size,
        "role": role,
    })

# Stage 01 controls always copied. Raw data remain license-gated unless explicitly author-verified.
s1_sums = parse_sha256sums(S1_ROOT / "SHA256SUMS.txt")
for name in ARTIFACT_CONTRACTS["01"]["control_artifacts"]:
    src = S1_ROOT / name
    if not src.is_file():
        hard_stop("Required Stage-01 control missing", {"artifact": name})
    expected = s1_sums.get(name)
    record_copy("01", src, RELEASE_ROOT / CANONICAL_STAGE_DIRS["01"] / name, expected,
                "frozen input control/provenance")

for rel, digest in s1_sums.items():
    src = S1_ROOT / rel
    if rel in ARTIFACT_CONTRACTS["01"]["control_artifacts"]:
        continue
    if RAW_DATA_REDISTRIBUTION_VERIFIED:
        record_copy("01", src, RELEASE_ROOT / CANONICAL_STAGE_DIRS["01"] / rel, digest,
                    "license-verified frozen dataset artifact")
    else:
        NOT_COPIED_RECORDS.append({
            "stage": "01", "path": rel, "sha256": digest, "size_bytes": src.stat().st_size,
            "reason": "RAW_DATA_REDISTRIBUTION_NOT_AUTHOR_VERIFIED",
        })

write_text(
    RELEASE_ROOT / CANONICAL_STAGE_DIRS["01"] / "RAW_DATA_NOT_REDISTRIBUTED.md",
    "# Raw data redistribution status\n\n"
    "The frozen Stage-01 data bytes were verified against the canonical checksum contract, "
    "but are not copied into the public core release because redistribution permissions require "
    "author verification. Exact source identities and hashes are preserved in the Stage-01 controls.\n"
)

# Stage 02: copy every checksum-contracted artifact.
s2_sums = parse_sha256sums(S2_ROOT / "P0_OUTPUT_SHA256SUMS.txt")
for rel, digest in s2_sums.items():
    record_copy("02", S2_ROOT / rel, RELEASE_ROOT / CANONICAL_STAGE_DIRS["02"] / rel, digest,
                "Stage-02 audit/provenance output")
# Include the checksum file itself.
record_copy("02", S2_ROOT / "P0_OUTPUT_SHA256SUMS.txt",
            RELEASE_ROOT / CANONICAL_STAGE_DIRS["02"] / "P0_OUTPUT_SHA256SUMS.txt",
            sha256_file(S2_ROOT / "P0_OUTPUT_SHA256SUMS.txt"), "Stage-02 outer checksum contract")

# Stage 03: full checksum contract except optional large artifacts.
for rel, digest in s3_sums.items():
    src = S3_ROOT / rel
    if rel in stage3_optional_paths:
        NOT_COPIED_RECORDS.append({
            "stage": "03", "path": rel, "sha256": digest, "size_bytes": src.stat().st_size,
            "reason": "OPTIONAL_LARGE_TRAINING_ARTIFACT",
        })
        if COPY_OPTIONAL_LARGE_TRAINING_ARTIFACTS:
            dst = RELEASE_ROOT / "models_optional" / rel
            record_copy("03", src, dst, digest, "optional full-training reproduction artifact")
        continue
    record_copy("03", src, RELEASE_ROOT / CANONICAL_STAGE_DIRS["03"] / rel, digest,
                "Stage-03 required core or small model artifact")
record_copy("03", S3_ROOT / "STAGE03_SHA256SUMS.txt",
            RELEASE_ROOT / CANONICAL_STAGE_DIRS["03"] / "STAGE03_SHA256SUMS.txt",
            S3_CHECKSUM_EXPECTED, "Stage-03 checksum contract")
if (S3_ROOT / "STAGE_STATUS.json").is_file():
    record_copy("03", S3_ROOT / "STAGE_STATUS.json",
                RELEASE_ROOT / CANONICAL_STAGE_DIRS["03"] / "STAGE_STATUS.json",
                sha256_file(S3_ROOT / "STAGE_STATUS.json"), "Stage-03 status")

# Because the preserved canonical Stage-03 checksum contract also references optional transformer
# artifacts, provide a separate core-only checksum contract for the files physically copied into 03_modeling.
s3_core_root = RELEASE_ROOT / CANONICAL_STAGE_DIRS["03"]
s3_core_lines = []
for p in sorted(s3_core_root.rglob("*"), key=str):
    if p.is_file() and p.name != "STAGE03_CORE_SHA256SUMS.txt":
        s3_core_lines.append(f"{sha256_file(p)}  {p.relative_to(s3_core_root).as_posix()}")
write_text(s3_core_root / "STAGE03_CORE_SHA256SUMS.txt", "\n".join(s3_core_lines) + "\n")

# Stages 04–09: copy all files named by their checksum contracts, then checksum/status files themselves.
checksum_specs = {
    "04": ("STAGE04_SHA256SUMS.txt", S4_CHECKSUM_EXPECTED, "STAGE_STATUS.json"),
    "05": ("STAGE05_SHA256SUMS.txt", EXPECTED_STAGE05["checksum_sha256"], "STAGE_STATUS.json"),
    "06": ("STAGE06_SHA256SUMS.txt", EXPECTED_STAGE06["checksum_sha256"], "STAGE_STATUS.json"),
    "07": ("STAGE07_SHA256SUMS.txt", EXPECTED_STAGE07["checksum_sha256"], "STAGE_STATUS.json"),
    "08": ("STAGE08_SHA256SUMS.txt", EXPECTED_STAGE08["checksum_sha256"], "STAGE08_STATUS.json"),
    "09": ("STAGE09_SHA256SUMS.txt", S9_IDENTITY[1], "STAGE_STATUS.json"),
}
for stage, (sum_name, sum_hash, status_name) in checksum_specs.items():
    root = STAGE_ROOTS[stage]
    sums = parse_sha256sums(root / sum_name)
    for rel, digest in sums.items():
        record_copy(stage, root / rel, RELEASE_ROOT / CANONICAL_STAGE_DIRS[stage] / rel, digest,
                    f"Stage-{stage} checksum-contracted output")
    record_copy(stage, root / sum_name, RELEASE_ROOT / CANONICAL_STAGE_DIRS[stage] / sum_name,
                sum_hash, f"Stage-{stage} checksum contract")
    if (root / status_name).is_file():
        expected_status = None
        if stage == "05": expected_status = EXPECTED_STAGE05["status_sha256"]
        elif stage == "06": expected_status = EXPECTED_STAGE06["status_sha256"]
        elif stage == "07": expected_status = EXPECTED_STAGE07["status_sha256"]
        elif stage == "08": expected_status = EXPECTED_STAGE08["status_sha256"]
        elif stage == "09": expected_status = S9_IDENTITY[2]
        record_copy(stage, root / status_name, RELEASE_ROOT / CANONICAL_STAGE_DIRS[stage] / status_name,
                    expected_status, f"Stage-{stage} status")

# Dedicated publication source-data and manuscript-table mirrors from verified Stage 08.
s8_release = RELEASE_ROOT / CANONICAL_STAGE_DIRS["08"]
for p in (s8_release / "figure_source_data").rglob("*") if (s8_release / "figure_source_data").exists() else []:
    if p.is_file():
        rel = p.relative_to(s8_release / "figure_source_data")
        safe_copy_verified(p, RELEASE_ROOT / "figure_source_data" / rel, sha256_file(p))
for base in [s8_release / "tables" / "main", s8_release / "tables" / "supplementary"]:
    if base.exists():
        for p in base.rglob("*"):
            if p.is_file():
                rel = Path(base.name) / p.name
                safe_copy_verified(p, RELEASE_ROOT / "manuscript_tables" / rel, sha256_file(p))

write_json(RELEASE_ROOT / "COPY_LEDGER.json", {
    "copied": COPIED_RECORDS,
    "not_copied": NOT_COPIED_RECORDS,
})
print("Verified artifact copying complete:", len(COPIED_RECORDS), "files")


Verified artifact copying complete: 245 files


## 8. Preserve canonical notebooks and archive historical/intermediate notebooks

In [9]:
for source_name in CANONICAL_NOTEBOOKS:
    src = NOTEBOOK_PATHS[source_name]
    dst = RELEASE_ROOT / "notebooks" / CANONICAL_NOTEBOOK_RELEASE_NAMES[source_name]
    # Copy the authenticated notebook byte-for-byte. Under CANONICAL_SOURCE matching the
    # observed whole-file SHA may differ only because the supplied notebook is an execution
    # instance; scientific source identity remains frozen and independently verified.
    safe_copy_verified(src, dst, NOTEBOOK_IDENTITY_AUDIT[source_name]["whole_notebook_sha256"])

archive_notice = (
    "HISTORICAL / NON-CANONICAL — DO NOT EXECUTE FOR REPRODUCTION\n\n"
    "Historical/intermediate notebooks are optional archival provenance and are not required "
    "for scientific reproduction. When an exact frozen whole-file SHA-256 match is mounted, "
    "that notebook is copied here; otherwise its absence is recorded in RELEASE_MANIFEST.json. "
    "The canonical scientific execution chain is exactly Stage 01 through Stage 09 using the "
    "nine notebooks in ../notebooks/.\n"
)
write_text(RELEASE_ROOT / "archive" / "historical_intermediate" / "README.md", "# Historical notebooks\n\n" + archive_notice)

for source_name in HISTORICAL_NOTEBOOKS:
    src = HISTORICAL_NOTEBOOK_PATHS.get(source_name)
    if src is None:
        continue
    dst = RELEASE_ROOT / "archive" / "historical_intermediate" / source_name
    safe_copy_verified(src, dst, HISTORICAL_NOTEBOOK_SHA256[source_name])

print(
    "Notebook release layout: PASS — exactly 9 canonical; historical/intermediate notebooks "
    "copied only when exact frozen whole-file matches are available"
)


Notebook release layout: PASS — exactly 9 canonical; historical/intermediate notebooks copied only when exact frozen whole-file matches are available


## 9. Stage locks and exact canonical identities

In [10]:

def file_hash_or_none(root, name):
    p = Path(root) / name
    return sha256_file(p) if p.is_file() else None

STAGE_LOCKS = {
    "01": {
        "manifest_sha256": file_hash_or_none(S1_ROOT, "ACQUISITION_MANIFEST.json"),
        "checksum_sha256": file_hash_or_none(S1_ROOT, "SHA256SUMS.txt"),
        "status_sha256": file_hash_or_none(S1_ROOT, "ACQUISITION_STATUS.json"),
        "protocol_version": "EXTERNAL-DATASET-BYTES-v1",
        "scientific_role": "frozen external dataset acquisition and byte identity",
        "notes": f"Canonical frozen ZIP provenance SHA256={EXPECTED_FROZEN_ZIP_SHA256}",
    },
    "02": {
        "manifest_sha256": file_hash_or_none(S2_ROOT, "P0_AUDIT_REPORT.json"),
        "checksum_sha256": file_hash_or_none(S2_ROOT, "P0_OUTPUT_SHA256SUMS.txt"),
        "status_sha256": None,
        "protocol_version": json.loads((S2_ROOT / "P0_RUN_METADATA.json").read_text(encoding="utf-8")).get("methodology_version"),
        "scientific_role": "P0 integrity/provenance/leakage audit",
        "notes": "Canonical Stage-02 decision may be P0_NO_GO; Stage-03 records the decontamination response and authoritative methodology identity.",
    },
    "03": {
        "manifest_sha256": file_hash_or_none(S3_ROOT, "MODEL_ARTIFACT_MANIFEST.json"),
        "checksum_sha256": S3_CHECKSUM_EXPECTED,
        "status_sha256": file_hash_or_none(S3_ROOT, "STAGE_STATUS.json"),
        "protocol_version": PROTOCOL_VERSION,
        "scientific_role": "primary model development on clean source splits",
        "notes": "Transformer checkpoints may be released separately; locked Stage-05 predictions are the downstream analysis interface.",
    },
    "04": {
        "manifest_sha256": file_hash_or_none(S4_ROOT, "CALIBRATION_ARTIFACT_MANIFEST.json"),
        "checksum_sha256": S4_CHECKSUM_EXPECTED,
        "status_sha256": file_hash_or_none(S4_ROOT, "STAGE_STATUS.json"),
        "protocol_version": PROTOCOL_VERSION,
        "scientific_role": "source DEV scalar temperature calibration",
        "notes": "",
    },
    "05": {
        "manifest_sha256": EXPECTED_STAGE05["manifest_sha256"],
        "checksum_sha256": EXPECTED_STAGE05["checksum_sha256"],
        "status_sha256": EXPECTED_STAGE05["status_sha256"],
        "protocol_version": PROTOCOL_VERSION,
        "scientific_role": "canonical external validation and H1/H2/H3/H3-P inference",
        "notes": EXPECTED_STAGE05["canonical_version"],
    },
    "06": {
        "manifest_sha256": EXPECTED_STAGE06["manifest_sha256"],
        "checksum_sha256": EXPECTED_STAGE06["checksum_sha256"],
        "status_sha256": EXPECTED_STAGE06["status_sha256"],
        "protocol_version": PROTOCOL_VERSION,
        "scientific_role": "canonical fixed-budget reliability evaluation",
        "notes": "",
    },
    "07": {
        "manifest_sha256": EXPECTED_STAGE07["manifest_sha256"],
        "checksum_sha256": EXPECTED_STAGE07["checksum_sha256"],
        "status_sha256": EXPECTED_STAGE07["status_sha256"],
        "protocol_version": PROTOCOL_VERSION,
        "scientific_role": "secondary/exploratory H3→H4 bridge",
        "notes": "Exploratory/non-causal; does not replace confirmatory inference.",
    },
    "08": {
        "manifest_sha256": EXPECTED_STAGE08["manifest_sha256"],
        "checksum_sha256": EXPECTED_STAGE08["checksum_sha256"],
        "status_sha256": EXPECTED_STAGE08["status_sha256"],
        "protocol_version": PROTOCOL_VERSION,
        "scientific_role": "audited FINAL2 manuscript figure/table reproduction",
        "notes": "Generic filename retained; identity is established by embedded manifest/checksum/status hashes.",
    },
    "09": {
        "manifest_sha256": S9_IDENTITY[0],
        "checksum_sha256": S9_IDENTITY[1],
        "status_sha256": S9_IDENTITY[2],
        "protocol_version": PROTOCOL_VERSION,
        "scientific_role": RELEASE_STAGE09_ROLE,
        "notes": f"Final source-site host-cluster sensitivity execution. Source manifest role string: {EXPECTED_STAGE09_SOURCE_ROLE}",
    },
}
write_json(RELEASE_ROOT / "STAGE_LOCKS.json", STAGE_LOCKS)
print("STAGE_LOCKS.json written.")


STAGE_LOCKS.json written.


## 10. Manuscript evidence matrix

In [11]:

EVIDENCE_ROWS = [
    {
        "claim_id": "H1", "claim": "Cross-platform discrimination/reliability transport",
        "hypothesis": "H1", "estimand": "AUROC change; relative NLL change; relative Balanced Brier change",
        "canonical_stage": "05", "canonical_artifact": "H1_RESULTS.json",
        "canonical_field": "rows; domain_model_seed_consensus; endpoint-specific estimability fields",
        "figure_id": "FIG02_H1_H2", "table_id": "TABLE02_H1_H2_SUMMARY",
        "stage09_sensitivity_artifact": "H1_H2_SOURCE_CLUSTER_SENSITIVITY.json",
        "claim_status": "USE_CANONICAL_STAGE05_STATUS",
        "safe_wording": "Report only the stored canonical endpoint/status language; preserve non-estimability where present.",
        "notes": "",
    },
    {
        "claim_id": "H2", "claim": "Transport/attenuation of source-trained calibration gain",
        "hypothesis": "H2", "estimand": "source vs target calibration gain",
        "canonical_stage": "05", "canonical_artifact": "H2_RESULTS.json",
        "canonical_field": "rows; domain_model_seed_consensus",
        "figure_id": "FIG02_H1_H2", "table_id": "TABLE02_H1_H2_SUMMARY",
        "stage09_sensitivity_artifact": "H1_H2_SOURCE_CLUSTER_SENSITIVITY.json",
        "claim_status": "USE_CANONICAL_STAGE05_STATUS",
        "safe_wording": "Use stored Stage-05 effect, CI/p-value, and estimability status without reinterpretation.",
        "notes": "",
    },
    {
        "claim_id": "H3_INDIVIDUAL", "claim": "Individual cue×platform transport effects",
        "hypothesis": "H3", "estimand": "ΔRD and cue-conditioned ΔNLL-gap",
        "canonical_stage": "05", "canonical_artifact": "H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv",
        "canonical_field": "delta_rd; delta_rd_ci95; delta_rd_p; ebm_delta_nll_gap_mean_5seeds; ebm_delta_nll_gap_ci95; ebm_delta_nll_gap_p; inferential_status",
        "figure_id": "FIG03_H3_CUE_TRANSPORT", "table_id": "TABLE03_H3_CUE_TRANSPORT",
        "stage09_sensitivity_artifact": "H3_SOURCE_CLUSTER_SENSITIVITY.json",
        "claim_status": "CANONICAL_STAGE05",
        "safe_wording": "Describe estimable contrasts only as stored; explicitly retain NE cells.",
        "notes": "Fixed 8 cue families × 4 platforms = 32-cell universe.",
    },
    {
        "claim_id": "H3_GLOBAL", "claim": "Global BanglaClick H3 outcome transport test",
        "hypothesis": "H3", "estimand": "32-dimensional global Wald test",
        "canonical_stage": "05", "canonical_artifact": "H3_RESULTS.json",
        "canonical_field": "global_banglaclick_outcome_wald",
        "figure_id": "FIG03_H3_CUE_TRANSPORT", "table_id": "TABLE03_H3_CUE_TRANSPORT",
        "stage09_sensitivity_artifact": "H3_SOURCE_CLUSTER_SENSITIVITY.json",
        "claim_status": "USE_STORED_GLOBAL_STATUS",
        "safe_wording": "Do not rescue or dimension-reduce a non-estimable canonical global test.",
        "notes": "",
    },
    {
        "claim_id": "H3_MULTIPLICITY", "claim": "Prespecified H3 family multiplicity control",
        "hypothesis": "H3", "estimand": "Simes family p-values and full-family BH-FDR",
        "canonical_stage": "05", "canonical_artifact": "H3_RESULTS.json",
        "canonical_field": "family_simes_outcome; family_simes_reliability; family_fdr_outcome_full; family_fdr_reliability_full",
        "figure_id": "FIG03_H3_CUE_TRANSPORT", "table_id": "TABLE03_H3_CUE_TRANSPORT",
        "stage09_sensitivity_artifact": "H3_SOURCE_CLUSTER_SENSITIVITY.json",
        "claim_status": "USE_FULL_PRESPECIFIED_FAMILY_STATUS",
        "safe_wording": "Do not run or imply BH-FDR on a post-hoc estimable subset.",
        "notes": "",
    },
    {
        "claim_id": "H3P", "claim": "Controlled cue-neutralization corroboration",
        "hypothesis": "H3-P", "estimand": "target-minus-source Δz",
        "canonical_stage": "05", "canonical_artifact": "H3_PERTURBATION_RESULTS.json",
        "canonical_field": "rows; inferential_status; target_minus_source_delta_z",
        "figure_id": "SUPP_H3P_*", "table_id": "SUPP_TABLE03_H3P",
        "stage09_sensitivity_artifact": "H3P_SOURCE_CLUSTER_SENSITIVITY.json",
        "claim_status": "CONTROLLED_CORROBORATION",
        "safe_wording": "Treat as controlled model-sensitivity corroboration, not a causal cue intervention.",
        "notes": "",
    },
    {
        "claim_id": "H4_PRIMARY", "claim": "Primary fixed 10% review-budget operational degradation",
        "hypothesis": "H4", "estimand": "target-minus-source lift, error capture, and AURC at primary budget",
        "canonical_stage": "06", "canonical_artifact": "H4_RESULTS.json",
        "canonical_field": "primary_budget; bootstrap_rows",
        "figure_id": "FIG04_H4_FIXED_BUDGET", "table_id": "TABLE04_H4_PRIMARY_10PCT_SUMMARY",
        "stage09_sensitivity_artifact": "H4_SOURCE_CLUSTER_SENSITIVITY.json",
        "claim_status": "CANONICAL_STAGE06",
        "safe_wording": "Use the fixed global 10% policy and stored metric-specific estimability statuses.",
        "notes": "",
    },
    {
        "claim_id": "H4_SENSITIVITY", "claim": "Fixed-budget sensitivity across prespecified budgets",
        "hypothesis": "H4", "estimand": "5%, 10%, 20% fixed-budget effects",
        "canonical_stage": "06", "canonical_artifact": "FIXED_BUDGET_TABLES.csv",
        "canonical_field": "budget; primary_budget; fixed-budget point metrics",
        "figure_id": "FIG04_H4_FIXED_BUDGET; SUPP_H4_*", "table_id": "SUPP_TABLE05_FULL_H4_BUDGET",
        "stage09_sensitivity_artifact": "H4_SOURCE_CLUSTER_SENSITIVITY.json",
        "claim_status": "CANONICAL_STAGE06_SENSITIVITY",
        "safe_wording": "Label 10% as primary and other budgets as sensitivity.",
        "notes": "",
    },
    {
        "claim_id": "BAITBUSTER", "claim": "Independent-collection H3 replication on BaitBuster human data",
        "hypothesis": "H3 replication", "estimand": "cue replication ΔRD and ΔNLL-gap",
        "canonical_stage": "05", "canonical_artifact": "H3_BAITBUSTER_REPLICATION_CORRECTED.csv",
        "canonical_field": "cue-level replication rows and inferential_status",
        "figure_id": "SUPP_BAITBUSTER_H3", "table_id": "SUPP_TABLE04_BAITBUSTER_H3",
        "stage09_sensitivity_artifact": "",
        "claim_status": "CANONICAL_STAGE05_REPLICATION",
        "safe_wording": "Describe as independent-collection replication/corroboration, preserving NE status.",
        "notes": "",
    },
    {
        "claim_id": "H3_H4_BRIDGE", "claim": "Exploratory H3→H4 association",
        "hypothesis": "secondary/exploratory", "estimand": "32-cell descriptive association and platform-FE slope",
        "canonical_stage": "07", "canonical_artifact": "H3_H4_BRIDGE_RESULTS.json",
        "canonical_field": "pairs; point; bootstrap.platform_fe_slope.estimability_status",
        "figure_id": "FIG05_H3_H4_BRIDGE", "table_id": "",
        "stage09_sensitivity_artifact": "",
        "claim_status": "EXPLORATORY_NON_CAUSAL",
        "safe_wording": "Descriptive/exploratory association only; no causal claim and no rescue of H3.",
        "notes": "Point-cell source: BRIDGE_32_CELL_POINT_TABLE.csv.",
    },
    {
        "claim_id": "LOO", "claim": "Leave-one-platform-out bridge robustness",
        "hypothesis": "secondary robustness", "estimand": "LOO platform-FE bridge point estimates",
        "canonical_stage": "07", "canonical_artifact": "LEAVE_ONE_PLATFORM_OUT_ROBUSTNESS.json",
        "canonical_field": "leave-one-platform-out robustness results",
        "figure_id": "SUPP_BRIDGE_LOO", "table_id": "SUPP_TABLE07_LOO_POINT",
        "stage09_sensitivity_artifact": "",
        "claim_status": "SECONDARY_ROBUSTNESS",
        "safe_wording": "Secondary robustness only; does not change the fixed four-platform primary bridge universe.",
        "notes": "",
    },
    {
        "claim_id": "STAGE09_SOURCE_SITE", "claim": "Source-site host-cluster sensitivity",
        "hypothesis": "sensitivity only", "estimand": "cluster-sensitive H1/H2/H3/H3-P/H4 re-evaluation",
        "canonical_stage": "09", "canonical_artifact": "CLAIM_CONCORDANCE_TABLE.csv",
        "canonical_field": "claim-level concordance classifications",
        "figure_id": "", "table_id": "",
        "stage09_sensitivity_artifact": "CLAIM_CONCORDANCE_TABLE.csv",
        "claim_status": "SENSITIVITY_ONLY",
        "safe_wording": RELEASE_STAGE09_ROLE,
        "notes": "Stage-09 sensitivity never upgrades or replaces canonical Stage-05/06/07/08 inference.",
    },
]

evidence_path = RELEASE_ROOT / "MANUSCRIPT_EVIDENCE_MATRIX.csv"
with evidence_path.open("w", encoding="utf-8", newline="") as f:
    fields = [
        "claim_id", "claim", "hypothesis", "estimand", "canonical_stage",
        "canonical_artifact", "canonical_field", "figure_id", "table_id",
        "stage09_sensitivity_artifact", "claim_status", "safe_wording", "notes",
    ]
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(EVIDENCE_ROWS)

# Every canonical evidence artifact must physically exist in the copied release.
for row in EVIDENCE_ROWS:
    stage = row["canonical_stage"]
    artifact = row["canonical_artifact"]
    p = RELEASE_ROOT / CANONICAL_STAGE_DIRS[stage] / artifact
    if not p.is_file():
        hard_stop("Evidence matrix points to missing canonical artifact",
                  {"claim_id": row["claim_id"], "path": str(p)})

print("MANUSCRIPT_EVIDENCE_MATRIX.csv: PASS")


MANUSCRIPT_EVIDENCE_MATRIX.csv: PASS


## 11. Environment, hardware, license, provenance, privacy and protocol documentation

In [12]:

def load_json_if(root, name):
    p = Path(root) / name
    if p.is_file():
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            return None
    return None

historical_env_records = {}
for stage, root in STAGE_ROOTS.items():
    candidates = ["RUN_METADATA.json", "P0_RUN_METADATA.json", "ENVIRONMENT_PREFLIGHT.json"]
    stage_records = {}
    for name in candidates:
        obj = load_json_if(root, name)
        if obj is not None:
            stage_records[name] = obj
    historical_env_records[stage] = stage_records

write_json(RELEASE_ROOT / "HISTORICALLY_RECORDED_ENVIRONMENT.json", historical_env_records)

assembly_packages = {}
for dist in importlib_metadata.distributions():
    try:
        name = dist.metadata["Name"]
        version = dist.version
        if name and version:
            assembly_packages[name] = version
    except Exception:
        continue
assembly_environment = {
    "label": "Stage10 assembly environment — NOT necessarily identical to historical stage environments.",
    "python": sys.version,
    "platform": platform.platform(),
    "packages": dict(sorted(assembly_packages.items(), key=lambda kv: kv[0].lower())),
}
write_json(RELEASE_ROOT / "STAGE10_ASSEMBLY_ENVIRONMENT.json", assembly_environment)

requirements_lines = [
    "# Stage10 assembly environment — NOT necessarily identical to historical Stage03–09 environments.",
    "# This file MUST NOT be described as a lockfile for the historical scientific runs.",
]
requirements_lines.extend(f"{name}=={version}" for name, version in sorted(assembly_packages.items(), key=lambda kv: kv[0].lower()))
write_text(RELEASE_ROOT / "requirements.txt", "\n".join(requirements_lines) + "\n")

env_md = [
    "# Environment",
    "",
    "## HISTORICALLY_RECORDED_ENVIRONMENT",
    "",
    "The release preserves the environment metadata that the canonical stages actually recorded.",
    "Stage 01/02 record Python 3.12.13 in the executed provenance; later stages preserve their own RUN_METADATA / preflight records where available.",
    "No missing historical package version is inferred from the Stage 10 environment.",
    "",
    "## STAGE10_ASSEMBLY_ENVIRONMENT",
    "",
    "The file `STAGE10_ASSEMBLY_ENVIRONMENT.json` and `requirements.txt` describe only the environment used to assemble this release.",
    "They are **not necessarily identical to the historical scientific stage environments**.",
]
write_text(RELEASE_ROOT / "ENVIRONMENT.md", "\n".join(env_md) + "\n")

gap_md = """# Environment gap

The canonical notebooks and stage outputs preserve substantial provenance, including Python/runtime and selected package metadata in several stages. However, the historical execution chain does **not** provide a single complete, immutable package lock covering every package and transitive dependency for Stages 03–09.

Therefore:

- `HISTORICALLY_RECORDED_ENVIRONMENT.json` contains only historically captured information.
- `STAGE10_ASSEMBLY_ENVIRONMENT.json` and `requirements.txt` describe the Stage 10 release-assembly runtime only.
- Missing historical package versions are not back-filled from the current environment.
- The Stage 10 `requirements.txt` MUST NOT be claimed to recreate the original Stage 03–09 software environments.
- Model, tokenizer, dataset, manifest, and checksum identities remain separately pinned where the original stages recorded them.

This gap is documentary, not a license to rerun or alter any scientific stage during release assembly.
"""
write_text(RELEASE_ROOT / "ENVIRONMENT_GAP.md", gap_md)

# Hardware records: preserve only what was actually captured.
hardware_lines = ["# Hardware", "", "Historically recorded hardware/runtime fields:"]
for stage, records in historical_env_records.items():
    found = []
    for name, obj in records.items():
        for key in ["cpu_count", "gpu_status", "runtime", "platform"]:
            if key in obj:
                found.append(f"- Stage {stage} / {name} / `{key}`: `{obj[key]}`")
    hardware_lines.extend(found)
hardware_lines += [
    "",
    "No unrecorded hardware characteristic is inferred.",
    "Stage 10 assembly itself requires no accelerator and performs no model/statistical computation.",
]
write_text(RELEASE_ROOT / "HARDWARE.md", "\n".join(hardware_lines) + "\n")

# Data/license documentation from the canonical frozen lock. Do not invent terms.
sources_lock = json.loads((S1_ROOT / "SOURCES_LOCK.json").read_text(encoding="utf-8"))
dataset_rows = []
datasets_obj = sources_lock.get("sources", sources_lock.get("datasets", sources_lock))
if isinstance(datasets_obj, dict):
    for name, rec in datasets_obj.items():
        if not isinstance(rec, dict):
            continue
        text = json.dumps(rec, ensure_ascii=False)
        repo = rec.get("repo_id") or rec.get("repo") or rec.get("source_url") or rec.get("authoritative", {}).get("source_url")
        revision = rec.get("revision") or rec.get("commit") or rec.get("requested_revision")
        license_note = rec.get("license_note")
        dataset_rows.append({
            "name": name,
            "source": rec.get("source_url") or rec.get("authoritative", {}).get("source_url") or "See SOURCES_LOCK.json",
            "repository": repo or "See SOURCES_LOCK.json",
            "revision": revision or "See SOURCES_LOCK.json",
            "hash": "See Stage-01 SHA256SUMS.txt / SOURCES_LOCK.json",
            "license": "REQUIRES AUTHOR VERIFICATION",
            "redistribution_permission": "REQUIRES AUTHOR VERIFICATION",
            "access_method": rec.get("kind") or "See SOURCES_LOCK.json",
            "restrictions": license_note or "REQUIRES AUTHOR VERIFICATION",
        })

data_license_lines = ["# Data license", "", "No license is invented by Stage 10.",
                      "Recorded source notes are preserved, but public redistribution remains gated by author verification.", ""]
for r in dataset_rows:
    data_license_lines += [
        f"## {r['name']}",
        f"- Source: {r['source']}",
        f"- Repository: {r['repository']}",
        f"- Revision: {r['revision']}",
        f"- Hash: {r['hash']}",
        f"- License: {r['license']}",
        f"- Redistribution permission: {r['redistribution_permission']}",
        f"- Access method: {r['access_method']}",
        f"- Restrictions / recorded note: {r['restrictions']}",
        "",
    ]
write_text(RELEASE_ROOT / "DATA_LICENSE.md", "\n".join(data_license_lines))

data_avail = """# Data availability

Stage 10 verifies the exact frozen Stage-01 data bytes against the canonical checksum contract. By default, raw dataset bytes are **not copied into the public core release** until redistribution permission is author-verified.

The release preserves:
- frozen source/revision identities;
- Stage-01 acquisition manifest, source lock, status and SHA-256 contract;
- exact BanglaBait TEST SHA-256 used by Stage 09;
- Stage-03 clean-data manifest and clean-split hashes;
- canonical Stage-05 locked prediction Parquets, which are the downstream analysis interface for the reported Stage-05/06/07 results.

Where a dataset cannot legally be redistributed, users must reacquire it from the recorded source and verify its hash. `RAW_DATA_REDISTRIBUTION_VERIFIED` must only be changed after author verification.
"""
write_text(RELEASE_ROOT / "DATA_AVAILABILITY.md", data_avail)

# Models and licenses
model_lock = json.loads((S3_ROOT / "MODEL_CONFIG_LOCK.json").read_text(encoding="utf-8"))
model_records = json.loads((S3_ROOT / "MODEL_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8")).get("models", [])
model_hash_index = defaultdict(list)
for rec in model_records:
    model_hash_index[str(rec.get("model"))].append({
        "seed": rec.get("seed"),
        "artifact_path": rec.get("artifact_path"),
        "artifact_state_sha256": rec.get("artifact_state_sha256"),
    })

model_specs = model_lock.get("models", {})
model_license_lines = [
    "# Model licenses", "",
    "Stage 10 does not infer licenses from model names, software packages, or repository defaults.",
    "Exact Stage-03 model/config identities are preserved; redistribution permissions remain subject to author verification.",
    "",
]
for key in ["ebm", "tfidf_lr", "banglabert", "xlmr"]:
    rec = model_specs.get(key, {})
    repo = rec.get("repo_id") or "N/A — locally trained model family"
    revision = rec.get("revision") or "N/A — see Stage-03 configuration lock"
    model_license_lines += [
        f"## {key}",
        f"- Source: {'pretrained repository + Stage-03 clean-source fine-tuning' if rec.get('repo_id') else 'Stage-03 clean-source training'}",
        f"- Repository: {repo}",
        f"- Revision: {revision}",
        f"- Hash: see `03_modeling/MODEL_ARTIFACT_MANIFEST.json`; recorded artifact-state entries: `{model_hash_index.get(key, [])}`",
        "- License: REQUIRES AUTHOR VERIFICATION",
        "- Redistribution permission: REQUIRES AUTHOR VERIFICATION",
        "- Access method: see `03_modeling/MODEL_CONFIG_LOCK.json` and canonical Stage-03 notebook",
        "- Restrictions: optional transformer checkpoint redistribution is license-dependent; the core release does not require those checkpoints for reported downstream numbers.",
        "",
    ]
write_text(RELEASE_ROOT / "MODEL_LICENSES.md", "\n".join(model_license_lines))

annotation_md = """# Annotation provenance

Annotation and dataset provenance are preserved through the canonical Stage-01 `SOURCES_LOCK.json`, `ACQUISITION_MANIFEST.json`, Stage-02 audit artifacts, and Stage-03 clean-data/decontamination manifests.

Stage 10 does not reinterpret, relabel, repair, or regenerate annotations. Any source-specific annotation license or redistribution term remains subject to the original publisher/repository and requires author verification before redistribution.
"""
write_text(RELEASE_ROOT / "ANNOTATION_PROVENANCE.md", annotation_md)

privacy_md = """# Privacy / PII

Stage 10 is a release-engineering pass. It does not create new participant data or infer personal attributes.

The anonymized release process:
- redacts local Kaggle owner path segments and email-address patterns from text artifacts;
- flags probable secrets/API keys;
- does not rewrite scientific source repository identifiers, dataset revisions, hashes, DOI identifiers, or model repository identifiers;
- avoids redistributing raw source datasets unless redistribution is author-verified.

Any unresolved identity-bearing URL or free-text field is reported for author review in `ANONYMIZATION_REPORT.json`.
"""
write_text(RELEASE_ROOT / "PRIVACY_PII.md", privacy_md)

protocol_md = """# Protocol

Canonical scientific protocol: `SWARABYANJAN-CROSSPLATFORM-v1.0.0`.

Canonical execution chain:
1. Stage 01 — frozen dataset acquisition / byte identity
2. Stage 02 — P0 integrity, provenance and leakage audit
3. Stage 03 — primary model development
4. Stage 04 — calibration
5. Stage 05 — canonical external validation + H1/H2/H3/H3-P
6. Stage 06 — fixed-budget reliability evaluation
7. Stage 07 — exploratory H3→H4 bridge
8. Stage 08 — audited FINAL2 manuscript figures/tables
9. Stage 09 — source-site host-cluster sensitivity only

Stage 10 is **not a scientific stage**. It assembles and verifies already-existing outputs only.
Stage 09 is sensitivity only and never replaces canonical Stages 05/06/07/08.
"""
write_text(RELEASE_ROOT / "PROTOCOL.md", protocol_md)

readme = """# Swarabyanjan Cross-Platform Paper Release

This package is assembled from the cryptographically verified outputs of the canonical nine-stage execution chain.

## Canonical chain
Exactly nine scientific stages are canonical: Stage 01 through Stage 09. Historical/intermediate Stage-05/06 notebooks are optional archival provenance, are copied under `archive/historical_intermediate/` only when exact frozen whole-file matches are mounted, and are not required for scientific reproduction.

## Downstream analysis interface
The locked Stage-05 prediction Parquets are the primary downstream analysis interface for reproduction of the reported Stage-05/06/07 results. Stage-03 transformer checkpoints are not required to reproduce the already reported downstream numbers and are therefore separated as optional full-training-reproduction artifacts.

## Environment
Historical environment metadata and Stage-10 assembly environment metadata are intentionally separated. See `ENVIRONMENT.md` and `ENVIRONMENT_GAP.md`.

## Licensing
No unverified license is asserted. See `DATA_LICENSE.md` and `MODEL_LICENSES.md`.

## Integrity
`STAGE_LOCKS.json`, `RELEASE_MANIFEST.json`, and `RELEASE_SHA256SUMS.txt` define the release identity.
"""
write_text(RELEASE_ROOT / "README.md", readme)

print("Documentation written without license/environment overclaim.")


Documentation written without license/environment overclaim.


## 12. Build anonymized submission-ready release

In [13]:

ANON_ROOT = RELEASE_ROOT / "ANONYMIZED_RELEASE"
ANON_ROOT.mkdir(parents=True, exist_ok=True)

# Copy release materials except raw-data gates, optional models, archive history, and the anonymized directory itself.
anon_exclude_top = {"ANONYMIZED_RELEASE", "models_optional", "archive"}
text_redaction_patterns = [
    (re.compile(r"(?i)(/kaggle/input/datasets/)[^/]+/"), r"\1<REDACTED_OWNER>/"),
    (re.compile(r"(?i)(/kaggle/input/notebooks/)[^/]+/"), r"\1<REDACTED_OWNER>/"),
    (re.compile(r"(?i)(/home/)[^/]+/"), r"\1<REDACTED_USER>/"),
    (re.compile(r"(?i)([A-Z0-9._%+-]+)@([A-Z0-9.-]+\.[A-Z]{2,})"), "<REDACTED_EMAIL>"),
]
secret_patterns = [
    re.compile(r"(?i)\b(?:api[_-]?key|secret|token|password)\b\s*[:=]\s*['\"][^'\"]+['\"]"),
    re.compile(r"\b(?:ghp_|hf_)[A-Za-z0-9]{20,}\b"),
]
url_pattern = re.compile(r"https?://[^\s)>\]\"']+")
KNOWN_KAGGLE_OWNER = "smalakarishere"
KNOWN_KAGGLE_NOTEBOOK_PATH = f"/kaggle/input/notebooks/{KNOWN_KAGGLE_OWNER}/"
KAGGLE_RESERVED_NONAUTHOR_PREFIXES = {
    "account", "api", "c", "code", "competitions", "datasets", "discussions", "docs",
    "kernels", "learn", "models", "notebooks", "organizations", "static", "usercontent",
}
kaggle_owner_scoped_url_pattern = re.compile(
    r"(?i)(https?://(?:www\.)?kaggle\.com/(?:code|kernels|notebooks)/)([^/\s?#]+)"
)
kaggle_profile_url_pattern = re.compile(
    r"(?i)(https?://(?:www\.)?kaggle\.com/)([^/\s?#]+)"
)
anonymization_report = {"redactions": [], "flags": [], "skipped": []}

def redact_author_identifying_kaggle_urls(text, rel):
    new = text

    def redact_owner_scoped(match):
        anonymization_report["flags"].append({
            "file": rel,
            "type": "KAGGLE_AUTHOR_URL_REDACTED",
            "value": "<REDACTED_KAGGLE_AUTHOR_URL>",
        })
        return match.group(1) + "<REDACTED_OWNER>"

    new, n_scoped = kaggle_owner_scoped_url_pattern.subn(redact_owner_scoped, new)
    if n_scoped:
        anonymization_report["redactions"].append({
            "file": rel,
            "pattern": "KAGGLE_OWNER_SCOPED_URL",
            "count": n_scoped,
        })

    def redact_profile(match):
        first_segment = match.group(2)
        if first_segment.lower() in KAGGLE_RESERVED_NONAUTHOR_PREFIXES:
            return match.group(0)
        anonymization_report["flags"].append({
            "file": rel,
            "type": "KAGGLE_AUTHOR_URL_REDACTED",
            "value": "<REDACTED_KAGGLE_AUTHOR_URL>",
        })
        return match.group(1) + "<REDACTED_OWNER>"

    before = new
    new = kaggle_profile_url_pattern.sub(redact_profile, new)
    n_profile = sum(
        1
        for match in kaggle_profile_url_pattern.finditer(before)
        if match.group(2).lower() not in KAGGLE_RESERVED_NONAUTHOR_PREFIXES
    )
    if n_profile:
        anonymization_report["redactions"].append({
            "file": rel,
            "pattern": "KAGGLE_PROFILE_URL",
            "count": n_profile,
        })
    return new

def redact_text(text, rel):
    new = text
    for pat, repl in text_redaction_patterns:
        newer, n = pat.subn(repl, new)
        if n:
            anonymization_report["redactions"].append({"file": rel, "pattern": pat.pattern, "count": n})
        new = newer
    new = redact_author_identifying_kaggle_urls(new, rel)
    for pat in secret_patterns:
        for m in pat.finditer(new):
            anonymization_report["flags"].append({"file": rel, "type": "POSSIBLE_SECRET", "value": "<REDACTED_IN_REPORT>"})
        new = pat.sub("<REDACTED_SECRET>", new)
    # Public scientific URLs are preserved. Author-identifying Kaggle URLs were redacted above.
    for u in sorted(set(url_pattern.findall(new))):
        if re.match(r"(?i)https?://(?:www\.)?kaggle\.com/(?:datasets|models)/", u):
            continue
        if "kaggle.com/" in u.lower() and "<redacted_owner" in u.lower():
            continue
        if not any(domain in u for domain in [
            "huggingface.co", "github.com/MBillahsust", "github.com/kamruzzaman15",
            "github.com/Shibu4064", "github.com/abdalimran", "data.mendeley.com",
            "kaggle.com/datasets/hrithikmajumdar",
        ]):
            anonymization_report["flags"].append({
                "file": rel, "type": "URL_REVIEW_FOR_AUTHOR_IDENTITY", "value": u
            })
    return new

for p in sorted(RELEASE_ROOT.rglob("*"), key=str):
    if not p.is_file():
        continue
    rel = p.relative_to(RELEASE_ROOT)
    if rel.parts[0] in anon_exclude_top:
        continue
    if rel.name in {"RELEASE_MANIFEST.json", "RELEASE_SHA256SUMS.txt", "RELEASE_STATUS.json"}:
        continue
    dst = ANON_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if is_probable_text_file(p):
        try:
            text = p.read_text(encoding="utf-8")
            dst.write_text(redact_text(text, rel.as_posix()), encoding="utf-8")
        except UnicodeDecodeError:
            shutil.copy2(p, dst)
    else:
        shutil.copy2(p, dst)

write_json(ANON_ROOT / "ANONYMIZATION_REPORT.json", anonymization_report)
write_text(
    ANON_ROOT / "README_ANONYMIZED.md",
    "# Anonymized submission release\n\n"
    "This tree is derived from the verified core release. Local identity-bearing paths and email patterns "
    "are redacted from text artifacts. Scientific provenance identifiers, hashes, dataset/model source identifiers, "
    "DOIs, and canonical stage identities are preserved. Review `ANONYMIZATION_REPORT.json` before submission.\n"
)

# Fail closed if any known author identity remains in anonymized text artifacts.
residual_identity_leaks = []
remaining_email_pattern = re.compile(r"(?i)[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}")
for p in sorted(ANON_ROOT.rglob("*"), key=str):
    if not p.is_file() or not is_probable_text_file(p):
        continue
    try:
        text = p.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        continue
    lower = text.lower()
    rel = p.relative_to(ANON_ROOT).as_posix()
    if KNOWN_KAGGLE_OWNER.lower() in lower:
        residual_identity_leaks.append({"file": rel, "type": "KNOWN_KAGGLE_USERNAME"})
    if KNOWN_KAGGLE_NOTEBOOK_PATH.lower() in lower:
        residual_identity_leaks.append({"file": rel, "type": "KNOWN_KAGGLE_NOTEBOOK_PATH"})
    if remaining_email_pattern.search(text):
        residual_identity_leaks.append({"file": rel, "type": "EMAIL_ADDRESS"})

if residual_identity_leaks:
    hard_stop("Anonymized release contains residual author-identifying text", residual_identity_leaks)
assert not residual_identity_leaks, "Anonymized release identity-leak assertion failed"

print("Anonymized release built; review flags:", len(anonymization_report["flags"]))


Anonymized release built; review flags: 2573


## 13. Final release manifest, outer SHA-256 contract, and QA

In [14]:
# Release manifest covers payload files before the final outer checksum/status controls are written.
payload_records = []
for p in sorted(RELEASE_ROOT.rglob("*"), key=str):
    if not p.is_file():
        continue
    rel = p.relative_to(RELEASE_ROOT).as_posix()
    if rel in {"RELEASE_MANIFEST.json", "RELEASE_SHA256SUMS.txt", "RELEASE_STATUS.json"}:
        continue
    payload_records.append({
        "path": rel,
        "size_bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    })

release_manifest = {
    "schema_version": RELEASE_SCHEMA_VERSION,
    "release_name": "swarabyanjan-crossplatform-paper",
    "created_utc": utc_now(),
    "scientific_computation_executed": False,
    "canonical_scientific_stage_count": 9,
    "stage01_input_form": STAGE01_INPUT_AUDIT,
    "canonical_notebooks": [
        {
            "stage": f"{i:02d}",
            "source_name": name,
            "release_name": CANONICAL_NOTEBOOK_RELEASE_NAMES[name],
            # Record the actual authenticated selected-notebook byte identity while retaining
            # the previously frozen whole-file identity separately as the expected identity.
            "sha256": NOTEBOOK_IDENTITY_AUDIT[name]["whole_notebook_sha256"],
            "expected_whole_file_sha256": CANONICAL_NOTEBOOK_SHA256[name],
            "whole_notebook_sha256": NOTEBOOK_IDENTITY_AUDIT[name]["whole_notebook_sha256"],
            "canonical_source_sha256": NOTEBOOK_IDENTITY_AUDIT[name]["canonical_source_sha256"],
            "cell_count": NOTEBOOK_IDENTITY_AUDIT[name]["cell_count"],
            "execution_state": NOTEBOOK_IDENTITY_AUDIT[name]["execution_state"],
            "match_type": NOTEBOOK_IDENTITY_AUDIT[name]["match_type"],
        }
        for i, name in enumerate(CANONICAL_NOTEBOOKS, start=1)
    ],
    "historical_intermediate_notebooks": [
        {
            "name": name,
            "expected_whole_file_sha256": HISTORICAL_NOTEBOOK_SHA256[name],
            "present": HISTORICAL_NOTEBOOK_AUDIT[name]["present"],
            "availability": HISTORICAL_NOTEBOOK_AUDIT[name]["availability"],
            "match_type": HISTORICAL_NOTEBOOK_AUDIT[name]["match_type"],
            "sha256": HISTORICAL_NOTEBOOK_AUDIT[name]["whole_notebook_sha256"],
            "label": "HISTORICAL / NON-CANONICAL — DO NOT EXECUTE FOR REPRODUCTION",
        }
        for name in HISTORICAL_NOTEBOOKS
    ],
    "stage_locks": STAGE_LOCKS,
    "payload_files": payload_records,
}
write_json(RELEASE_ROOT / "RELEASE_MANIFEST.json", release_manifest)

# QA checks
qa = {}

qa["exactly_9_canonical_stages"] = len(CANONICAL_NOTEBOOKS) == 9 and len(list((RELEASE_ROOT / "notebooks").glob("*.ipynb"))) == 9
qa["historical_notebooks_not_canonical"] = all(
    not (RELEASE_ROOT / "notebooks" / name).exists() for name in HISTORICAL_NOTEBOOKS
)
qa["stage05_hashes_unchanged"] = (
    sha256_file(S5_ROOT / "STAGE05_ARTIFACT_MANIFEST.json") == EXPECTED_STAGE05["manifest_sha256"]
    and sha256_file(S5_ROOT / "STAGE05_SHA256SUMS.txt") == EXPECTED_STAGE05["checksum_sha256"]
    and sha256_file(S5_ROOT / "STAGE_STATUS.json") == EXPECTED_STAGE05["status_sha256"]
)
qa["stage06_hashes_unchanged"] = (
    sha256_file(S6_ROOT / "REPRODUCIBILITY_MANIFEST.json") == EXPECTED_STAGE06["manifest_sha256"]
    and sha256_file(S6_ROOT / "STAGE06_SHA256SUMS.txt") == EXPECTED_STAGE06["checksum_sha256"]
    and sha256_file(S6_ROOT / "STAGE_STATUS.json") == EXPECTED_STAGE06["status_sha256"]
)
qa["stage07_hashes_unchanged"] = (
    sha256_file(S7_ROOT / "REPRODUCIBILITY_MANIFEST.json") == EXPECTED_STAGE07["manifest_sha256"]
    and sha256_file(S7_ROOT / "STAGE07_SHA256SUMS.txt") == EXPECTED_STAGE07["checksum_sha256"]
    and sha256_file(S7_ROOT / "STAGE_STATUS.json") == EXPECTED_STAGE07["status_sha256"]
)
qa["stage08_hashes_unchanged"] = (
    sha256_file(S8_ROOT / "MANIFEST.json") == EXPECTED_STAGE08["manifest_sha256"]
    and sha256_file(S8_ROOT / "STAGE08_SHA256SUMS.txt") == EXPECTED_STAGE08["checksum_sha256"]
    and sha256_file(S8_ROOT / "STAGE08_STATUS.json") == EXPECTED_STAGE08["status_sha256"]
)
qa["stage09_identity_preserved"] = (
    sha256_file(S9_ROOT / "REPRODUCIBILITY_MANIFEST.json") == S9_IDENTITY[0]
    and sha256_file(S9_ROOT / "STAGE09_SHA256SUMS.txt") == S9_IDENTITY[1]
    and sha256_file(S9_ROOT / "STAGE_STATUS.json") == S9_IDENTITY[2]
)
qa["all_required_stage_roots_found"] = set(STAGE_ROOTS) == {f"{i:02d}" for i in range(1, 10)}
qa["all_stage_prediction_parquets_present"] = all(
    (RELEASE_ROOT / CANONICAL_STAGE_DIRS["05"] / name).is_file()
    for name in EXPECTED_STAGE05["prediction_sha256"]
)
qa["all_stage_controls_present"] = all(
    all((STAGE_ROOTS[stage] / rel).is_file() for rel in spec["control_artifacts"])
    for stage, spec in ARTIFACT_CONTRACTS.items()
)
qa["figure_table_source_mapping_preserved"] = (
    (RELEASE_ROOT / CANONICAL_STAGE_DIRS["08"] / "FIGURE_SOURCE_COVERAGE.json").is_file()
    and (RELEASE_ROOT / "figure_source_data").is_dir()
    and (RELEASE_ROOT / "manuscript_tables").is_dir()
)
qa["stage09_sensitivity_only"] = (
    STAGE_LOCKS["09"]["scientific_role"] == RELEASE_STAGE09_ROLE
)
qa["no_scientific_computation_executed"] = True

# Notebook preservation is two-level: copied bytes must equal the authenticated selected input,
# and every copied notebook must still satisfy the frozen canonical source identity + cell count.
def _released_canonical_notebook_identity_ok(name):
    p = RELEASE_ROOT / "notebooks" / CANONICAL_NOTEBOOK_RELEASE_NAMES[name]
    if not p.is_file():
        return False
    observed = inspect_notebook_path(p, expected_cell_count=CANONICAL_NOTEBOOK_CELL_COUNTS[name])
    return (
        observed.get("parse_ok")
        and observed.get("whole_notebook_sha256") == NOTEBOOK_IDENTITY_AUDIT[name]["whole_notebook_sha256"]
        and observed.get("cell_count") == CANONICAL_NOTEBOOK_CELL_COUNTS[name]
        and observed.get("canonical_source_sha256") == CANONICAL_NOTEBOOK_SOURCE_SHA256[name]
    )

qa["no_canonical_notebook_modified"] = all(
    _released_canonical_notebook_identity_ok(name) for name in CANONICAL_NOTEBOOKS
)
qa["canonical_notebook_source_identity_preserved"] = all(
    NOTEBOOK_IDENTITY_AUDIT[name]["canonical_source_sha256"] == CANONICAL_NOTEBOOK_SOURCE_SHA256[name]
    and NOTEBOOK_IDENTITY_AUDIT[name]["cell_count"] == CANONICAL_NOTEBOOK_CELL_COUNTS[name]
    and NOTEBOOK_IDENTITY_AUDIT[name]["match_type"] in {"WHOLE_FILE", "CANONICAL_SOURCE"}
    for name in CANONICAL_NOTEBOOKS
)
qa["copied_artifacts_hash_verified"] = all(
    (RELEASE_ROOT / rec["release_relative"]).is_file()
    and sha256_file(RELEASE_ROOT / rec["release_relative"]) == rec["sha256"]
    for rec in COPIED_RECORDS
)
qa["stage03_optional_large_artifacts_documented"] = (RELEASE_ROOT / "OPTIONAL_LARGE_TRAINING_ARTIFACTS.csv").is_file()
qa["environment_gap_documented"] = (RELEASE_ROOT / "ENVIRONMENT_GAP.md").is_file()
qa["license_gaps_documented"] = (
    (RELEASE_ROOT / "DATA_LICENSE.md").is_file()
    and (RELEASE_ROOT / "MODEL_LICENSES.md").is_file()
)
qa["manuscript_evidence_matrix_present"] = (RELEASE_ROOT / "MANUSCRIPT_EVIDENCE_MATRIX.csv").is_file()

failed = [k for k, v in qa.items() if not v]
release_status = {
    "status": "PASS" if not failed else "FAIL",
    "timestamp_utc": utc_now(),
    "scientific_computation_executed": False,
    "qa": qa,
    "failed_checks": failed,
}
write_json(RELEASE_ROOT / "RELEASE_STATUS.json", release_status)
if failed:
    hard_stop("Release QA failed", {"failed_checks": failed})

# Outer checksum contract includes every release file except itself.
outer_lines = []
for p in sorted(RELEASE_ROOT.rglob("*"), key=str):
    if not p.is_file():
        continue
    rel = p.relative_to(RELEASE_ROOT).as_posix()
    if rel == "RELEASE_SHA256SUMS.txt":
        continue
    outer_lines.append(f"{sha256_file(p)}  {rel}")
write_text(RELEASE_ROOT / "RELEASE_SHA256SUMS.txt", "\n".join(outer_lines) + "\n")

# Verify the just-written outer contract without modifying any covered file.
verify_checksum_contract(RELEASE_ROOT, "RELEASE_SHA256SUMS.txt", require_all=True)

print("FINAL RELEASE QA: PASS")
print("Release root:", RELEASE_ROOT)


FINAL RELEASE QA: PASS
Release root: /kaggle/working/swarabyanjan-crossplatform-paper


## End state

A successful future execution produces a release package with:
- exactly nine canonical scientific notebooks;
- up to three optional historical/intermediate notebooks, copied only when exact frozen whole-file matches are mounted;
- cryptographically verified stage artifacts;
- Stage 03 optional-large training artifact inventory;
- release documentation and environment/license gaps;
- manuscript evidence mapping;
- anonymized submission tree;
- outer release manifest/checksum/status controls.

**No scientific stage is rerun by this notebook.**
